This notebok includes the baseline generation according to [the README file](https://github.com/bowang-lab/MedSAM/tree/LiteMedSAM) in the LiteMedSAM branch.
The main goal of this notebook is creating a based and tested baseline model for us to later compare to in our project.

In [ ]:
# Daily setup cell

import os
from google.colab import drive
from google.colab import userdata

# Mount Google Drive
drive.mount('/content/drive/')

# --- CONFIGURATION ---
project_folder = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/"
repo_owner = "LinoyHarush"
repo_name = "MedSAM_enh"
# ---------------------

# Securely fetch the user's personal PAT
try:
    pat = userdata.get('GITHUB_PAT')
except userdata.SecretNotFoundError:
    print("Error: 'GITHUB_PAT' not found in Colab Secrets.")
    print("Please add it via the key icon on the left sidebar before running.")
    raise

repo_url = f"https://{pat}@github.com/{repo_owner}/{repo_name}.git"
repo_path = os.path.join(project_folder, repo_name)

# Navigate to the working folder
os.makedirs(project_folder, exist_ok=True)
%cd {project_folder}

# Clone or Pull
if not os.path.isdir(repo_name):
    print(f"Cloning {repo_name} into Drive for the first time...")
    !git clone {repo_url}
else:
    print(f"Directory {repo_name} exists. Updating repository...")
    %cd {repo_name}
    # Update the origin URL so Git uses the PAT of whoever is running this cell right now
    !git remote set-url origin {repo_url}
    !git pull
    %cd ..

# Enter the repository for the remainder of the session
%cd {repo_name}

In [ ]:
"Git Pull"

%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM/MedSAM_enh/
#!git stash
!git pull origin main

In [ ]:
# Install the packages and their dependencies
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
!pip install -e .

In [ ]:
" Dont Run Again "

# 1. Navigate to your specified repository root
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM/MedSAM_enh/

# 2. Fetch the latest history from GitHub
!git fetch origin

# 3. Force your local Drive folder to perfectly match the main branch, discarding all phantom changes
!git reset --hard origin/main

# 4. Pull to ensure everything is fully synced
!git pull origin main

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

In [ ]:
# Ensure the directory is in drive
import os
print(os.path.ismount('/content/drive'))
print(os.path.exists(project_folder))

In [ ]:
# Local git identity for this session so commits have Shelly's name
!git config --global user.name "<your-name>"
!git config --global user.email "<your-email>"

In [ ]:
# Local git identity for this session so commits have Linoy's name
!git config --global user.name "<your-name>"
!git config --global user.email "<your-email>"

In [ ]:
test_output_folder = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/test_output"

#Sanity test and Compute Metrics

In [ ]:
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
# Reccomanded Sanity Test
!python CVPR24_LiteMedSAM_infer.py -i /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/test_demo/imgs/ -o /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/test_demo/new_segs

In [ ]:
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
# Compute Metrics
!python evaluation/compute_metrics.py -s /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/test_demo/new_segs -g /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/test_demo/gts -csv_dir /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/test_demo/metrics.csv

In [ ]:
# Metric Satistics
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/test_demo/

import pandas as pd
import matplotlib.pyplot as plt

# Load your metrics
df = pd.read_csv('metrics.csv')

# Clean up the case names so the chart labels are easier to read
df['case'] = df['case'].str.replace('_demo.npz', '', regex=False)
df['case'] = df['case'].str.replace('2DBox_', '', regex=False)
df['case'] = df['case'].str.replace('3DBox_', '', regex=False)

# Set the case column as the index for plotting
df.set_index('case', inplace=True)

# Generate the grouped bar chart
# Slightly wider figure for better label spacing
ax = df[['dsc', 'nsd']].plot(kind='bar', figsize=(14, 7), color=['#4C72B0', '#55A868'], width=0.8)

plt.title('LiteMedSAM Inference Metrics by Modality', fontsize=16, pad=20)
plt.ylabel('Score (0.0 to 1.0)', fontsize=12)
plt.xlabel('Imaging Modality', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=11)

# Extended y-axis limit to 1.15 to make room for the text labels on top
plt.ylim(0, 1.15)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Move legend slightly out of the way of the data
plt.legend(['Dice Similarity Coefficient (DSC)', 'Normalized Surface Distance (NSD)'], loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2)

# Add data labels on top of each bar
for container in ax.containers:
    # fmt='%.3f' formats the number to 3 decimal places
    ax.bar_label(container, fmt='%.3f', padding=3, fontsize=10)

plt.tight_layout()
plt.savefig(f'{test_output_folder}/test_demo_original_checkpoints/metrics_bar_chart.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# Check Demo Visually
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/

import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import SimpleITK as sitk

# Define directories
img_dir = 'test_demo/imgs'
seg_dir = 'test_demo/new_segs'

# Safely extract image and mask data regardless of the file format
def load_case(filename):
    img_path = os.path.join(img_dir, filename)
    seg_path = os.path.join(seg_dir, filename)

    try:
        # Handle NumPy zip arrays (Demo default)
        if filename.endswith('.npz'):
            img_data = np.load(img_path)
            seg_data = np.load(seg_path)
            # Extract the first array found in the npz file
            img = img_data[img_data.files[0]]
            mask = seg_data[seg_data.files[0]]
            return img, mask

        # Handle standard 2D images
        elif filename.endswith(('.png', '.jpg', '.jpeg')):
            img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
            mask = cv2.imread(seg_path, cv2.IMREAD_GRAYSCALE)
            return img, mask

        # Handle 3D NIfTI volumes using SimpleITK
        elif filename.endswith(('.nii.gz', '.nii')):
            img = sitk.GetArrayFromImage(sitk.ReadImage(img_path))
            mask = sitk.GetArrayFromImage(sitk.ReadImage(seg_path))
            return img, mask
    except Exception as e:
        print(f"Error loading {filename}: {e}")

    return None, None

# Sort files into 2D and 3D based on the filenames
all_files = sorted(os.listdir(seg_dir))
files_2d = [f for f in all_files if '2D' in f or f.endswith('.png')]
files_3d = [f for f in all_files if '3D' in f or f.endswith('.nii.gz')]

# --- Plot 1: 2D Modalities ---
if files_2d:
    fig_2d, axes_2d = plt.subplots(2, 4, figsize=(20, 10))
    fig_2d.suptitle('2D Segmentation Masks by Modality', fontsize=18, fontweight='bold', y=1.02)
    axes_2d = axes_2d.flatten()

    for i, filename in enumerate(files_2d):
        if i >= 8: break
        img, mask = load_case(filename)

        if img is not None:
            # Display image and overlay the mask
            axes_2d[i].imshow(img, cmap='gray')
            axes_2d[i].imshow(mask, cmap='jet', alpha=0.4)

            # Clean up title for readability
            title = filename.replace('2DBox_', '').replace('_demo.npz', '').replace('.png', '')
            axes_2d[i].set_title(title, fontsize=14)
        axes_2d[i].axis('off')

    plt.tight_layout()
    plt.savefig(f'{test_output_folder}/test_demo_original_checkpoints/2D_Segmentation_Masks.png', bbox_inches='tight', dpi=300)
    plt.show()

# --- Plot 2: 3D Modalities (Middle Slices) ---
if files_3d:
    fig_3d, axes_3d = plt.subplots(1, 2, figsize=(12, 6))
    fig_3d.suptitle('3D Segmentation Masks (Middle Slices)', fontsize=18, fontweight='bold', y=1.05)

    for i, filename in enumerate(files_3d):
        if i >= 2: break
        img, mask = load_case(filename)

        if img is not None:
            # Extract the middle slice of the 3D volume
            mid_idx = img.shape[0] // 2 if img.ndim == 3 else 0

            # Extract 2D slice from 3D array
            slice_img = img[mid_idx] if img.ndim == 3 else img
            slice_mask = mask[mid_idx] if mask.ndim == 3 else mask

            axes_3d[i].imshow(slice_img, cmap='gray')
            axes_3d[i].imshow(slice_mask, cmap='jet', alpha=0.4)

            title = filename.replace('3DBox_', '').replace('_demo.npz', '').replace('.nii.gz', '')
            axes_3d[i].set_title(f"{title}\n(Slice {mid_idx})", fontsize=14)
        axes_3d[i].axis('off')

    plt.tight_layout()
    plt.savefig(f'{test_output_folder}/test_demo_original_checkpoints/3D_Segmentation_Masks.png', bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:
# Error Map for Demo
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import SimpleITK as sitk

# Directories
img_dir = 'test_demo/imgs'
gt_dir = 'test_demo/gts'
pred_dir = 'test_demo/new_segs'

# Helper function to load arrays safely
def get_array(path):
    if not os.path.exists(path): return None
    try:
        if path.endswith('.npz'):
            data = np.load(path)
            return data[data.files[0]]
        elif path.endswith(('.png', '.jpg', '.jpeg')):
            if 'imgs' in path:
                return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
            else:
                return cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        elif path.endswith(('.nii.gz', '.nii')):
            return sitk.GetArrayFromImage(sitk.ReadImage(path))
    except Exception as e:
        print(f"Error loading {path}: {e}")
    return None

def plot_comparisons(file_list, category_name):
    if not file_list: return

    # Create a figure with 4 columns per modality
    fig, axes = plt.subplots(len(file_list), 4, figsize=(22, 5 * len(file_list)))

    # Handle single row indexing edge-case
    if len(file_list) == 1: axes = [axes]

    # Add a custom master legend for the colors
    tp_patch = mpatches.Patch(color='yellow', label='Match (True Positive)')
    fn_patch = mpatches.Patch(color='lime', label='Missed (False Negative)')
    fp_patch = mpatches.Patch(color='red', label='Over-predicted (False Positive)')
    fig.legend(handles=[tp_patch, fn_patch, fp_patch], loc='upper center', ncol=3, fontsize=14, bbox_to_anchor=(0.5, 0.98))
    fig.suptitle(f"{category_name} Modalities - Ground Truth vs Prediction", fontsize=20, fontweight='bold', y=1.02)

    for i, filename in enumerate(file_list):
        img = get_array(os.path.join(img_dir, filename))
        gt = get_array(os.path.join(gt_dir, filename))
        pred = get_array(os.path.join(pred_dir, filename))

        if img is None or gt is None or pred is None:
            continue

        # Handle 3D slicing (Distinguish from 2D RGB images by checking the last channel)
        if img.ndim == 3 and img.shape[-1] != 3:
            mid = img.shape[0] // 2
            img, gt, pred = img[mid], gt[mid], pred[mid]

        # Standardize masks to binary (0 and 1)
        gt_bin = (gt > 0).astype(np.uint8)
        pred_bin = (pred > 0).astype(np.uint8)

        # Build the RGB Difference Map
        diff_map = np.zeros((*gt_bin.shape, 3), dtype=np.uint8)
        diff_map[(gt_bin == 1) & (pred_bin == 1)] = [255, 255, 0] # Yellow
        diff_map[(gt_bin == 1) & (pred_bin == 0)] = [0, 255, 0]   # Lime Green
        diff_map[(gt_bin == 0) & (pred_bin == 1)] = [255, 0, 0]   # Red

        # Clean title string
        clean_title = filename.replace('2DBox_', '').replace('3DBox_', '').replace('_demo.npz', '').replace('.nii.gz', '')
        row_axes = axes[i]

        # Col 1: Original
        row_axes[0].imshow(img, cmap='gray')
        row_axes[0].set_title(f"{clean_title}\nOriginal Image", fontsize=14)

        # Col 2: Ground Truth
        row_axes[1].imshow(img, cmap='gray')
        # Using a custom colormap overlay to visualize GT
        row_axes[1].imshow(np.ma.masked_where(gt_bin == 0, gt_bin), cmap='autumn', alpha=0.6)
        row_axes[1].set_title("Ground Truth", fontsize=14)

        # Col 3: Prediction
        row_axes[2].imshow(img, cmap='gray')
        row_axes[2].imshow(np.ma.masked_where(pred_bin == 0, pred_bin), cmap='cool', alpha=0.6)
        row_axes[2].set_title("Model Prediction", fontsize=14)

        # Col 4: Difference Map (Overlay)
        row_axes[3].imshow(img, cmap='gray')
        # Mask out the background [0,0,0] so we only overlay the active errors/matches
        diff_mask = np.any(diff_map != [0, 0, 0], axis=-1)
        row_axes[3].imshow(np.ma.masked_where(~diff_mask, diff_map[..., 0]), cmap='gray') # Dummy for masking
        row_axes[3].imshow(diff_map, alpha=0.5)
        row_axes[3].set_title("Difference Map", fontsize=14)

        for ax in row_axes:
            ax.axis('off')

    plt.tight_layout()
    plt.subplots_adjust(top=0.92)

    # Save the file dynamically based on whether it is 2D or 3D
    plt.savefig(f'{test_output_folder}/test_demo_original_checkpoints/{category_name}_Difference_Maps.png', bbox_inches='tight', dpi=300)

    plt.show()

# Fetch and sort the files
all_files = sorted(os.listdir(pred_dir))
files_2d = [f for f in all_files if '2D' in f or f.endswith('.png')]
files_3d = [f for f in all_files if '3D' in f or f.endswith('.nii.gz')]

# Execute plotting
print("Rendering 2D Modalities...")
plot_comparisons(files_2d, "2D")

print("Rendering 3D Modalities (Middle Slices)...")
plot_comparisons(files_3d, "3D")

#Model Training
##Data preprocessing

In [ ]:
# Ensure we're in the project root
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh

# Create the new directory structure for our custom work - we stored the downloaded lite_medsam.pth in weights
!mkdir -p experiments/weights
!mkdir -p experiments/checkpoints
!mkdir -p data

In [ ]:
# Install the missing 3D connected components library
!pip install connected-components-3d

# Run the pre-processing script to convert the dataset to npz format
!python MedSAM/pre_CT_MR.py \
    -img_path data/FLARE22Train/images \
    -img_name_suffix _0000.nii.gz \
    -gt_path data/FLARE22Train/labels \
    -gt_name_suffix .nii.gz \
    -output_path data \
    -num_workers 2 \
    -modality CT \
    -anatomy Abd \
    -window_level 40 \
    -window_width 400 \
    --save_nii

In [ ]:
# Check successful execution
print("Training files generated:")
!ls data/MedSAM_train/CT_Abd/ | grep .npz | wc -l

print("\nTesting files generated:")
!ls data/MedSAM_test/CT_Abd/ | grep .npz | wc -l

In [ ]:
# Convert the training npz to npy format for training
!python MedSAM/npz_to_npy.py \
    -npz_dir data/MedSAM_train/CT_Abd \
    -npy_dir data/npy \
    -num_workers 2

In [ ]:
# Ensure we are in the project root
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh

# Download Kvasir-Instruments, ignoring SSL certificate errors (-k)
print("Downloading Kvasir-Instruments...")
!curl -k -L -A "Mozilla/5.0 (Windows NT 10.0; Win64; x64)" "https://datasets.simula.no/downloads/kvasir-instrument.zip" -o data/kvasir_instrument.zip
print("Extracting Kvasir-Instruments...")
!unzip -q data/kvasir_instrument.zip -d data/kvasir_instruments/

# Download Kvasir-SEG, ignoring SSL certificate errors (-k)
print("Downloading Kvasir-SEG...")
!curl -k -L -A "Mozilla/5.0 (Windows NT 10.0; Win64; x64)" "https://datasets.simula.no/downloads/kvasir-seg.zip" -o data/kvasir_seg.zip
print("Extracting Kvasir-SEG...")
!unzip -q data/kvasir_seg.zip -d data/kvasir_seg/

# Final Cleanup
!rm data/kvasir_instrument.zip data/kvasir_seg.zip
print("Kvasir datasets successfully added!")

In [ ]:
# Check the contents and file sizes of the data folder
!ls -lh data/

In [ ]:
"Failed cell - to delete later"

# Ensure you are in the project root
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh

# Execute the 2D preprocessing script on kvasir-seg
!python MedSAM/pre_grey_rgb.py \
    -img_path data/kvasir_seg/images \
    -img_name_suffix .jpg \
    -gt_path data/kvasir_seg/masks \
    -gt_name_suffix .jpg \
    -output_path data \
    -num_workers 2

In [ ]:
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh

# Create target directories for npz and npy outputs
!mkdir -p data/kvasir_seg/npz
!mkdir -p data/kvasir_seg/npy
!mkdir -p data/kvasir_instruments/npz
!mkdir -p data/kvasir_instruments/npy

!echo "Directories confirmed and ready."

In [ ]:
# Extract kvasir-instrument files
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_instruments

# Extract the nested archives
!echo "Extracting images..."
!tar -xzf images.tar.gz

!echo "Extracting masks..."
!tar -xzf masks.tar.gz

# Clean up the compressed files to save Drive space
!rm images.tar.gz masks.tar.gz

!echo "Extraction complete! Folders are now ready."

In [ ]:
%%writefile preprocess_2d.py
import os
import argparse
import numpy as np
from skimage import io
from tqdm import tqdm
from pathlib import Path

def parse_args():
    parser = argparse.ArgumentParser(description="Preprocess 2D RGB datasets for MedSAM")
    parser.add_argument("-img_dir", type=str, required=True, help="Path to the raw images directory")
    parser.add_argument("-gt_dir", type=str, required=True, help="Path to the ground truth masks directory")
    parser.add_argument("-output_dir", type=str, required=True, help="Path to save the output .npz files")
    parser.add_argument("-prefix", type=str, default="MedSAM_2D_", help="Prefix for the output files")
    return parser.parse_args()

def main():
    args = parse_args()

    os.makedirs(args.output_dir, exist_ok=True)

    image_files = sorted([f for f in os.listdir(args.img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f"Found {len(image_files)} images in {args.img_dir}")

    success_count = 0
    for img_name in tqdm(image_files, desc=f"Processing {args.prefix.strip('_')}"):
        stem = Path(img_name).stem

        # Locate corresponding mask
        gt_name = None
        for ext in ['.jpg', '.jpeg', '.png']:
            candidate = os.path.join(args.gt_dir, stem + ext)
            if os.path.exists(candidate):
                gt_name = candidate
                break

        if gt_name is None:
            print(f"Warning: No matching mask found for {img_name}")
            continue

        # 1. Format Image (Force 3-channel RGB uint8)
        img = io.imread(os.path.join(args.img_dir, img_name))
        if img.ndim == 2:
            img = np.repeat(np.expand_dims(img, -1), 3, -1)
        elif img.shape[-1] > 3:
            img = img[:, :, :3]

        if np.max(img) > 255.0:
            img = np.uint8((img - img.min()) / (np.max(img) - np.min(img)) * 255.0)
        else:
            img = img.astype(np.uint8)

        # 2. Format Mask (Force 2D {0, 1} binary array via 128 threshold)
        gt = io.imread(gt_name)
        if gt.ndim == 3:
            gt = gt[:, :, 0]
        gt_binary = np.uint8(gt > 128)

        # 3. Save as compressed .npz
        save_path = os.path.join(args.output_dir, f"{args.prefix}{stem}.npz")
        np.savez_compressed(save_path, imgs=img, gts=gt_binary)
        success_count += 1

    print(f"\nSuccessfully generated {success_count} .npz files in: {args.output_dir}")

if __name__ == "__main__":
    main()

In [ ]:
" DONT RUN THIS , DELETE LATER"

# Execute our 2D preprocessing script on the two kvasir datasets
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh

!echo "--- Processing Kvasir-SEG ---"
!python preprocess_2d.py \
    -img_dir data/kvasir_seg/images \
    -gt_dir data/kvasir_seg/masks \
    -output_dir data/kvasir_seg/npz \
    -prefix "Endoscopy_Polyp_"

!echo -e "\n--- Processing Kvasir-Instruments ---"
!python preprocess_2d.py \
    -img_dir data/kvasir_instruments/images \
    -gt_dir data/kvasir_instruments/masks \
    -output_dir data/kvasir_instruments/npz \
    -prefix "Endoscopy_Instruments_"

# note: we get npz files from this run

In [ ]:
" DONT RUN THIS , DELETE LATER"


# We convert the kvasir npz files to npy files
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh

!echo "Converting Kvasir-SEG to .npy..."
!mkdir -p data/kvasir_seg/npy
!python MedSAM/npz_to_npy.py -npz_dir data/kvasir_seg/npz -npy_dir data/kvasir_seg/npy -num_workers 2

!echo "Converting Kvasir-Instruments to .npy..."
!mkdir -p data/kvasir_instruments/npy
!python MedSAM/npz_to_npy.py -npz_dir data/kvasir_instruments/npz -npy_dir data/kvasir_instruments/npy -num_workers 2

In [ ]:
" DONT RUN THIS , DELETE LATER"


"We need to separate to test and train before running this cell"
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
# Recommend Sanity Test on Kvasir-SEG
!python CVPR24_LiteMedSAM_infer.py -i /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_seg/images/ -o /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/experiments/kvasir_seg/og_checkpoints
# Recommend Sanity Test on Kvasir-Instrument
!python CVPR24_LiteMedSAM_infer.py -i /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_instrument/images/ -o /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/experiments/kvasir_instrument/og_checkpoints

In [ ]:
%%writefile split_kvasir.py

import os
import json
import shutil
from pathlib import Path

# --- Configuration ---
# Update this to point to the specific kvasir dataset folder you are splitting
base_dir = Path("/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_seg")

# --- 1. Auto-Detect Bounding Box Format ---
bb_mode = None
master_json_path = None
bbox_dir = base_dir / "bbox"

# Detect JSON format first
json_candidates = list(base_dir.glob("*.json"))
if json_candidates:
    bb_mode = "json"
    master_json_path = json_candidates[0]
    print(f"Detected JSON bounding box format: {master_json_path.name}")
    with open(master_json_path, "r") as f:
        master_bboxes = json.load(f)

# Detect CSV format in the explicitly named 'bbox' folder
elif bbox_dir.exists() and list(bbox_dir.glob("*.csv")):
    bb_mode = "csv"
    print(f"Detected CSV bounding box format in folder: {bbox_dir.name}")
else:
    print("Warning: No bounding box annotations (JSON or CSV in 'bbox' folder) detected.")

# --- 2. Process Splits ---
# Automatically processes train, val, and test if their respective .txt files exist
for split in ["train", "val", "test"]:
    split_file = base_dir / f"{split}.txt"
    if not split_file.exists():
        continue

    print(f"\nProcessing [{split.upper()}] split...")

    # Set up destination folders
    out_dir = base_dir / split
    out_images = out_dir / "images"
    out_masks = out_dir / "masks"
    out_images.mkdir(parents=True, exist_ok=True)
    out_masks.mkdir(parents=True, exist_ok=True)

    # Replicate the bbox folder structure for CSVs
    out_bboxes = None
    if bb_mode == "csv":
        out_bboxes = out_dir / "bbox"
        out_bboxes.mkdir(parents=True, exist_ok=True)

    # Read image IDs (stripping spaces and file extensions)
    with open(split_file, "r") as f:
        file_ids = [line.strip().split(".")[0] for line in f if line.strip()]

    filtered_bboxes = {}
    copied_img = 0
    copied_mask = 0
    copied_csv = 0

    for file_id in file_ids:
        # Locate and copy Image (glob handles both .jpg and .png automatically)
        img_candidates = list((base_dir / "images").glob(f"{file_id}.*"))
        if img_candidates:
            shutil.copy2(img_candidates[0], out_images / img_candidates[0].name)
            copied_img += 1

        # Locate and copy Mask (glob handles both .jpg and .png automatically)
        mask_candidates = list((base_dir / "masks").glob(f"{file_id}.*"))
        if mask_candidates:
            shutil.copy2(mask_candidates[0], out_masks / mask_candidates[0].name)
            copied_mask += 1

        # Handle Bounding Boxes based on detected mode
        if bb_mode == "json" and file_id in master_bboxes:
            filtered_bboxes[file_id] = master_bboxes[file_id]
        elif bb_mode == "csv":
            src_csv = bbox_dir / f"{file_id}.csv"
            if src_csv.exists():
                shutil.copy2(src_csv, out_bboxes / src_csv.name)
                copied_csv += 1

    # Save final bounding box annotations
    if bb_mode == "json":
        out_json = out_dir / "bounding_boxes.json"
        with open(out_json, "w") as f:
            json.dump(filtered_bboxes, f, indent=2)
        print(f"[{split.upper()}] Done! Images: {copied_img}, Masks: {copied_mask}, JSON BBoxes: {len(filtered_bboxes)}")
    elif bb_mode == "csv":
        print(f"[{split.upper()}] Done! Images: {copied_img}, Masks: {copied_mask}, CSV BBoxes: {copied_csv}")

print("\nDataset split complete.")

In [ ]:
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/

# Create train subfolder
!python split_kvasir.py \
  --split_file data/kvasir_instruments/train.txt \
  --data_dir data/kvasir_instruments \
  --output_dir data/kvasir_instruments/train

# Create test subfolder
!python split_kvasir.py \
  --split_file data/kvasir_instruments/test.txt \
  --data_dir data/kvasir_instruments \
  --output_dir data/kvasir_instruments/test

In [ ]:
!echo "Train images:" $(ls -1 /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_instruments/train/images | wc -l)
!echo "Train masks: " $(ls -1 /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_instruments/train/masks | wc -l)
!echo "Test images: " $(ls -1 /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_instruments/test/images | wc -l)
!echo "Test masks:  " $(ls -1 /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_instruments/test/masks | wc -l)

In [ ]:
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/

# Create train subfolder
!python split_kvasir.py \
  --split_file data/kvasir_seg/train.txt \
  --data_dir data/kvasir_seg \
  --output_dir data/kvasir_seg/train

# Create test subfolder
!python split_kvasir.py \
  --split_file data/kvasir_seg/val.txt \
  --data_dir data/kvasir_seg \
  --output_dir data/kvasir_instruments/test

In [ ]:
!echo "Train images:" $(ls -1 /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_seg/train/images | wc -l)
!echo "Train masks: " $(ls -1 /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_seg/train/masks | wc -l)
!echo "Test images: " $(ls -1 /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_seg/test/images | wc -l)
!echo "Test masks:  " $(ls -1 /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_seg/test/masks | wc -l)

In [ ]:
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/

## preprocess kvasir_instrument

!python MedSAM/pre_rgb_bbox_datasets.py \
  -output_path data/kvasir_instruments \
  -modality RGB \
  -anatomy KvasirInstruments \
  -img_name_suffix .jpg \
  -gt_name_suffix .png \
  -bbox_format json \
  -train_img_path data/kvasir_instruments/train/images \
  -train_gt_path data/kvasir_instruments/train/masks \
  -train_bbox_json data/kvasir_instruments/train/bounding_boxes.json \
  -test_img_path data/kvasir_instruments/test/images \
  -test_gt_path data/kvasir_instruments/test/masks \
  -test_bbox_json data/kvasir_instruments/test/bounding_boxes.json \
  --binary_mask


In [ ]:
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/

## preprocess kvasir_seg

!python MedSAM/pre_rgb_bbox_datasets.py \
  -output_path data/kvasir_seg \
  -modality RGB \
  -anatomy KvasirSeg \
  -img_name_suffix .jpg \
  -gt_name_suffix .jpg \
  -bbox_format csv_dir \
  -train_img_path data/kvasir_seg/train/images \
  -train_gt_path data/kvasir_seg/train/masks \
  -train_bbox_csv_dir data/kvasir_seg/train/bbox \
  -test_img_path data/kvasir_seg/test/images \
  -test_gt_path data/kvasir_seg/test/masks \
  -test_bbox_csv_dir data/kvasir_seg/test/bbox \
  --binary_mask

In [ ]:
### inference on kvasir instruments test on original lite medsam checkpoint ###

%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/
# Reccomanded Sanity Test
!python MedSAM/CVPR24_LiteMedSAM_infer.py -i /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_instruments/MedSAM_test/RGB_KvasirInstruments/imgs/ -o /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_instruments/MedSAM_test/RGB_KvasirInstruments/segs_og_ckpt

In [ ]:
### inference on kvasir seg test on original lite medsam checkpoint ###

%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
# Reccomanded Sanity Test
!python CVPR24_LiteMedSAM_infer.py -i /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_seg/MedSAM_test/RGB_KvasirSeg/imgs/ -o /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_seg/MedSAM_test/RGB_KvasirSeg/segs_og_ckpt

In [ ]:
### metric compute on kvasir instruments test on original lite medsam checkpoint ###

%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/
# Compute Metrics
!python MedSAM/evaluation/compute_metrics.py -s data/kvasir_instruments/MedSAM_test/RGB_KvasirInstruments/segs_og_ckpt -g data/kvasir_instruments/MedSAM_test/RGB_KvasirInstruments/gts -csv_dir data/kvasir_instruments/MedSAM_test/RGB_KvasirInstruments/metrics.csv


In [ ]:
### metric compute on kvasir seg test on original lite medsam checkpoint ###

%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/

# Compute Metrics
!python evaluation/compute_metrics.py -s data/kvasir_seg/MedSAM_test/RGB_KvasirSeg/segs_og_ckpt -g data/kvasir_seg/MedSAM_test/RGB_KvasirSeg/gts -csv_dir data/kvasir_seg/MedSAM_test/RGB_KvasirSeg/metrics.csv


In [ ]:
### metrics distribution of kvasir instruments on original lite med sam checkpoint

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the path to your metrics.csv file
csv_path = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/kvasir_instruments/MedSAM_test/RGB_KvasirInstruments/metrics_og_ckpt.csv"

# Load the dataset
df = pd.read_csv(csv_path)

# 1. Calculate and print the requested statistics
metrics = ['dsc', 'nsd']
print("--- Inference Statistics ---")
for metric in metrics:
    avg_val = df[metric].mean()
    med_val = df[metric].median()
    std_val = df[metric].std()

    print(f"\nMetric: {metric.upper()}")
    print(f"  Average (Mean): {avg_val:.4f}")
    print(f"  Median:         {med_val:.4f}")
    print(f"  Std Dev:        {std_val:.4f}")

# 2. Create the visualizations
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, metric in enumerate(metrics):
    # Boxplot shows the median, quartiles, and identifies outliers
    sns.boxplot(
        y=df[metric],
        ax=axes[i],
        color="lightgray",
        width=0.4,
        showfliers=False # Hide default outliers to prevent double-plotting with stripplot
    )

    # Stripplot overlays the actual distribution of all data points
    sns.stripplot(
        y=df[metric],
        ax=axes[i],
        color=sns.color_palette("husl", 2)[i],
        alpha=0.6,
        jitter=True,
        size=5
    )

    axes[i].set_title(f"Distribution of {metric.upper()}", fontsize=14, fontweight='bold')
    axes[i].set_ylabel("Score", fontsize=12)
    axes[i].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# Metric Satistics
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/test_demo/

import pandas as pd
import matplotlib.pyplot as plt

# Load your metrics
df = pd.read_csv('metrics.csv')

# Clean up the case names so the chart labels are easier to read
df['case'] = df['case'].str.replace('_demo.npz', '', regex=False)
df['case'] = df['case'].str.replace('2DBox_', '', regex=False)
df['case'] = df['case'].str.replace('3DBox_', '', regex=False)

# Set the case column as the index for plotting
df.set_index('case', inplace=True)

# Generate the grouped bar chart
# Slightly wider figure for better label spacing
ax = df[['dsc', 'nsd']].plot(kind='bar', figsize=(14, 7), color=['#4C72B0', '#55A868'], width=0.8)

plt.title('LiteMedSAM Inference Metrics by Modality', fontsize=16, pad=20)
plt.ylabel('Score (0.0 to 1.0)', fontsize=12)
plt.xlabel('Imaging Modality', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=11)

# Extended y-axis limit to 1.15 to make room for the text labels on top
plt.ylim(0, 1.15)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Move legend slightly out of the way of the data
plt.legend(['Dice Similarity Coefficient (DSC)', 'Normalized Surface Distance (NSD)'], loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2)

# Add data labels on top of each bar
for container in ax.containers:
    # fmt='%.3f' formats the number to 3 decimal places
    ax.bar_label(container, fmt='%.3f', padding=3, fontsize=10)

plt.tight_layout()
plt.savefig(f'{test_output_folder}/test_demo_original_checkpoints/metrics_bar_chart.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# Check Demo Visually
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/

import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import SimpleITK as sitk

# Define directories
img_dir = 'test_demo/imgs'
seg_dir = 'test_demo/new_segs'

# Safely extract image and mask data regardless of the file format
def load_case(filename):
    img_path = os.path.join(img_dir, filename)
    seg_path = os.path.join(seg_dir, filename)

    try:
        # Handle NumPy zip arrays (Demo default)
        if filename.endswith('.npz'):
            img_data = np.load(img_path)
            seg_data = np.load(seg_path)
            # Extract the first array found in the npz file
            img = img_data[img_data.files[0]]
            mask = seg_data[seg_data.files[0]]
            return img, mask

        # Handle standard 2D images
        elif filename.endswith(('.png', '.jpg', '.jpeg')):
            img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
            mask = cv2.imread(seg_path, cv2.IMREAD_GRAYSCALE)
            return img, mask

        # Handle 3D NIfTI volumes using SimpleITK
        elif filename.endswith(('.nii.gz', '.nii')):
            img = sitk.GetArrayFromImage(sitk.ReadImage(img_path))
            mask = sitk.GetArrayFromImage(sitk.ReadImage(seg_path))
            return img, mask
    except Exception as e:
        print(f"Error loading {filename}: {e}")

    return None, None

# Sort files into 2D and 3D based on the filenames
all_files = sorted(os.listdir(seg_dir))
files_2d = [f for f in all_files if '2D' in f or f.endswith('.png')]
files_3d = [f for f in all_files if '3D' in f or f.endswith('.nii.gz')]

# --- Plot 1: 2D Modalities ---
if files_2d:
    fig_2d, axes_2d = plt.subplots(2, 4, figsize=(20, 10))
    fig_2d.suptitle('2D Segmentation Masks by Modality', fontsize=18, fontweight='bold', y=1.02)
    axes_2d = axes_2d.flatten()

    for i, filename in enumerate(files_2d):
        if i >= 8: break
        img, mask = load_case(filename)

        if img is not None:
            # Display image and overlay the mask
            axes_2d[i].imshow(img, cmap='gray')
            axes_2d[i].imshow(mask, cmap='jet', alpha=0.4)

            # Clean up title for readability
            title = filename.replace('2DBox_', '').replace('_demo.npz', '').replace('.png', '')
            axes_2d[i].set_title(title, fontsize=14)
        axes_2d[i].axis('off')

    plt.tight_layout()
    plt.savefig(f'{test_output_folder}/test_demo_original_checkpoints/2D_Segmentation_Masks.png', bbox_inches='tight', dpi=300)
    plt.show()

# --- Plot 2: 3D Modalities (Middle Slices) ---
if files_3d:
    fig_3d, axes_3d = plt.subplots(1, 2, figsize=(12, 6))
    fig_3d.suptitle('3D Segmentation Masks (Middle Slices)', fontsize=18, fontweight='bold', y=1.05)

    for i, filename in enumerate(files_3d):
        if i >= 2: break
        img, mask = load_case(filename)

        if img is not None:
            # Extract the middle slice of the 3D volume
            mid_idx = img.shape[0] // 2 if img.ndim == 3 else 0

            # Extract 2D slice from 3D array
            slice_img = img[mid_idx] if img.ndim == 3 else img
            slice_mask = mask[mid_idx] if mask.ndim == 3 else mask

            axes_3d[i].imshow(slice_img, cmap='gray')
            axes_3d[i].imshow(slice_mask, cmap='jet', alpha=0.4)

            title = filename.replace('3DBox_', '').replace('_demo.npz', '').replace('.nii.gz', '')
            axes_3d[i].set_title(f"{title}\n(Slice {mid_idx})", fontsize=14)
        axes_3d[i].axis('off')

    plt.tight_layout()
    plt.savefig(f'{test_output_folder}/test_demo_original_checkpoints/3D_Segmentation_Masks.png', bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:


# # --- Configuration Paths ---
# base_dir = '/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data'
# dataset_name = 'kvasir_instruments' # Change to 'kvasir_seg' when you switch datasets

# img_dir  = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/imgs'
# gt_dir   = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/gts'
# pred_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/segs_og_ckpt'
# out_file = f'/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/experiments/{dataset_name}/og_ckpt/Difference_Maps.png'

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import SimpleITK as sitk
import random

# --- Configuration ---
base_dir = '/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data'
dataset_name = 'kvasir_instruments' # Change to 'kvasir_seg' when you switch datasets

# Directories
img_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/imgs'
gt_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/gts'
pred_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/segs_og_ckpt'

# Define and create the missing output folder
test_output_folder = f'/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/experiments/{dataset_name}/og_ckpt'
out_path = f'{test_output_folder}/Difference_Maps_10_ex2.png'
# os.makedirs(f'{test_output_folder}', exist_ok=True)

# Helper function to load arrays safely
def get_array(path):
    if not os.path.exists(path): return None
    try:
        if path.endswith('.npz'):
            data = np.load(path)
            return data[data.files[0]]
        elif path.endswith(('.png', '.jpg', '.jpeg')):
            if 'imgs' in path or 'images' in path:
                return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
            else:
                return cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        elif path.endswith(('.nii.gz', '.nii')):
            return sitk.GetArrayFromImage(sitk.ReadImage(path))
    except Exception as e:
        print(f"Error loading {path}: {e}")
    return None

def plot_comparisons(file_list, category_name, num_samples=None):
    if not file_list:
        print(f"No files found for {category_name}.")
        return

    # Random sampling to limit output
    if num_samples and len(file_list) > num_samples:
        print(f"Sampling {num_samples} files out of {len(file_list)}...")
        file_list = random.sample(file_list, num_samples)

    # Create a figure with 4 columns per modality
    fig, axes = plt.subplots(len(file_list), 4, figsize=(22, 5 * len(file_list)))

    # Handle single row indexing edge-case
    if len(file_list) == 1: axes = [axes]

    # Add a custom master legend for the colors
    tp_patch = mpatches.Patch(color='yellow', label='Match (True Positive)')
    fn_patch = mpatches.Patch(color='lime', label='Missed (False Negative)')
    fp_patch = mpatches.Patch(color='red', label='Over-predicted (False Positive)')
    fig.legend(handles=[tp_patch, fn_patch, fp_patch], loc='upper center', ncol=3, fontsize=14, bbox_to_anchor=(0.5, 0.98))
    fig.suptitle(f"{dataset_name} - {category_name} Modalities - Ground Truth vs Prediction", fontsize=20, fontweight='bold', y=1.02)

    for i, filename in enumerate(file_list):
        img = get_array(os.path.join(img_dir, filename))
        gt = get_array(os.path.join(gt_dir, filename))
        pred = get_array(os.path.join(pred_dir, filename))

        if img is None or gt is None or pred is None:
            continue

        # Handle 3D slicing (Distinguish from 2D RGB images by checking the last channel)
        if img.ndim == 3 and img.shape[-1] != 3:
            mid = img.shape[0] // 2
            img, gt, pred = img[mid], gt[mid], pred[mid]

        # Standardize masks to binary (0 and 1)
        gt_bin = (gt > 0).astype(np.uint8)
        pred_bin = (pred > 0).astype(np.uint8)

        # Build the RGB Difference Map
        diff_map = np.zeros((*gt_bin.shape, 3), dtype=np.uint8)
        diff_map[(gt_bin == 1) & (pred_bin == 1)] = [255, 255, 0] # Yellow
        diff_map[(gt_bin == 1) & (pred_bin == 0)] = [0, 255, 0]   # Lime Green
        diff_map[(gt_bin == 0) & (pred_bin == 1)] = [255, 0, 0]   # Red

        # Clean title string
        clean_title = filename.replace('2DBox_', '').replace('3DBox_', '').replace('_demo.npz', '').replace('.nii.gz', '').replace('.npz', '')
        row_axes = axes[i]

        # Col 1: Original
        row_axes[0].imshow(img, cmap='gray')
        row_axes[0].set_title(f"{clean_title}\nOriginal Image", fontsize=14)

        # Col 2: Ground Truth
        row_axes[1].imshow(img, cmap='gray')
        # Using a custom colormap overlay to visualize GT
        row_axes[1].imshow(np.ma.masked_where(gt_bin == 0, gt_bin), cmap='autumn', alpha=0.6)
        row_axes[1].set_title("Ground Truth", fontsize=14)

        # Col 3: Prediction
        row_axes[2].imshow(img, cmap='gray')
        row_axes[2].imshow(np.ma.masked_where(pred_bin == 0, pred_bin), cmap='cool', alpha=0.6)
        row_axes[2].set_title("Model Prediction", fontsize=14)

        # Col 4: Difference Map (Overlay)
        row_axes[3].imshow(img, cmap='gray')
        # Mask out the background [0,0,0] so we only overlay the active errors/matches
        diff_mask = np.any(diff_map != [0, 0, 0], axis=-1)
        row_axes[3].imshow(np.ma.masked_where(~diff_mask, diff_map[..., 0]), cmap='gray') # Dummy for masking
        row_axes[3].imshow(diff_map, alpha=0.5)
        row_axes[3].set_title("Difference Map", fontsize=14)

        for ax in row_axes:
            ax.axis('off')

    plt.tight_layout()
    plt.subplots_adjust(top=0.92)

    # Save the file dynamically to the newly created folder
    # out_path = f'{test_output_folder}/Difference_Maps.png'
    plt.savefig(out_path, bbox_inches='tight', dpi=300)
    print(f"Saved visualization to: {out_path}")

    plt.show()

if not os.path.exists(pred_dir):
    print(f"Prediction directory not found: {pred_dir}")
else:
    # Fetch and sort the files
    all_files = sorted(os.listdir(pred_dir))

    # Categorize files based on original logic, keeping .npz and .nii.gz intact
    files_2d = [f for f in all_files if '2D' in f or f.endswith(('.png', '.jpg', '.jpeg', '.npz')) and '3D' not in f]
    files_3d = [f for f in all_files if '3D' in f or f.endswith('.nii.gz')]

    # Select the number of samples to output here
    SAMPLES_TO_VIEW = 10

    # Execute plotting
    print("Rendering 2D Modalities...")
    plot_comparisons(files_2d, "2D", num_samples=SAMPLES_TO_VIEW)

    print("Rendering 3D Modalities (Middle Slices)...")
    plot_comparisons(files_3d, "3D", num_samples=SAMPLES_TO_VIEW)

In [ ]:


# # --- Configuration Paths ---
# base_dir = '/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data'
# dataset_name = 'kvasir_instruments' # Change to 'kvasir_seg' when you switch datasets

# img_dir  = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/imgs'
# gt_dir   = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/gts'
# pred_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/segs_og_ckpt'
# out_file = f'/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/experiments/{dataset_name}/og_ckpt/Difference_Maps.png'

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import SimpleITK as sitk
import random

# --- Configuration ---
base_dir = '/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data'
dataset_name = 'kvasir_seg' # Change to 'kvasir_seg' when you switch datasets

# Directories
img_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirSeg/imgs'
gt_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirSeg/gts'
pred_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirSeg/segs_og_ckpt'

# Define and create the missing output folder
test_output_folder = f'/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/experiments/{dataset_name}/og_ckpt'
out_path = f'{test_output_folder}/Difference_Maps_10_ex1.png'
# os.makedirs(f'{test_output_folder}', exist_ok=True)

# Helper function to load arrays safely
def get_array(path):
    if not os.path.exists(path): return None
    try:
        if path.endswith('.npz'):
            data = np.load(path)
            return data[data.files[0]]
        elif path.endswith(('.png', '.jpg', '.jpeg')):
            if 'imgs' in path or 'images' in path:
                return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
            else:
                return cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        elif path.endswith(('.nii.gz', '.nii')):
            return sitk.GetArrayFromImage(sitk.ReadImage(path))
    except Exception as e:
        print(f"Error loading {path}: {e}")
    return None

def plot_comparisons(file_list, category_name, num_samples=None):
    if not file_list:
        print(f"No files found for {category_name}.")
        return

    # Random sampling to limit output
    if num_samples and len(file_list) > num_samples:
        print(f"Sampling {num_samples} files out of {len(file_list)}...")
        file_list = random.sample(file_list, num_samples)

    # Create a figure with 4 columns per modality
    fig, axes = plt.subplots(len(file_list), 4, figsize=(22, 5 * len(file_list)))

    # Handle single row indexing edge-case
    if len(file_list) == 1: axes = [axes]

    # Add a custom master legend for the colors
    tp_patch = mpatches.Patch(color='yellow', label='Match (True Positive)')
    fn_patch = mpatches.Patch(color='lime', label='Missed (False Negative)')
    fp_patch = mpatches.Patch(color='red', label='Over-predicted (False Positive)')
    fig.legend(handles=[tp_patch, fn_patch, fp_patch], loc='upper center', ncol=3, fontsize=14, bbox_to_anchor=(0.5, 0.98))
    fig.suptitle(f"{dataset_name} - {category_name} Modalities - Ground Truth vs Prediction", fontsize=20, fontweight='bold', y=1.02)

    for i, filename in enumerate(file_list):
        img = get_array(os.path.join(img_dir, filename))
        gt = get_array(os.path.join(gt_dir, filename))
        pred = get_array(os.path.join(pred_dir, filename))

        if img is None or gt is None or pred is None:
            continue

        # Handle 3D slicing (Distinguish from 2D RGB images by checking the last channel)
        if img.ndim == 3 and img.shape[-1] != 3:
            mid = img.shape[0] // 2
            img, gt, pred = img[mid], gt[mid], pred[mid]

        # Standardize masks to binary (0 and 1)
        gt_bin = (gt > 0).astype(np.uint8)
        pred_bin = (pred > 0).astype(np.uint8)

        # Build the RGB Difference Map
        diff_map = np.zeros((*gt_bin.shape, 3), dtype=np.uint8)
        diff_map[(gt_bin == 1) & (pred_bin == 1)] = [255, 255, 0] # Yellow
        diff_map[(gt_bin == 1) & (pred_bin == 0)] = [0, 255, 0]   # Lime Green
        diff_map[(gt_bin == 0) & (pred_bin == 1)] = [255, 0, 0]   # Red

        # Clean title string
        clean_title = filename.replace('2DBox_', '').replace('3DBox_', '').replace('_demo.npz', '').replace('.nii.gz', '').replace('.npz', '')
        row_axes = axes[i]

        # Col 1: Original
        row_axes[0].imshow(img, cmap='gray')
        row_axes[0].set_title(f"{clean_title}\nOriginal Image", fontsize=14)

        # Col 2: Ground Truth
        row_axes[1].imshow(img, cmap='gray')
        # Using a custom colormap overlay to visualize GT
        row_axes[1].imshow(np.ma.masked_where(gt_bin == 0, gt_bin), cmap='autumn', alpha=0.6)
        row_axes[1].set_title("Ground Truth", fontsize=14)

        # Col 3: Prediction
        row_axes[2].imshow(img, cmap='gray')
        row_axes[2].imshow(np.ma.masked_where(pred_bin == 0, pred_bin), cmap='cool', alpha=0.6)
        row_axes[2].set_title("Model Prediction", fontsize=14)

        # Col 4: Difference Map (Overlay)
        row_axes[3].imshow(img, cmap='gray')
        # Mask out the background [0,0,0] so we only overlay the active errors/matches
        diff_mask = np.any(diff_map != [0, 0, 0], axis=-1)
        row_axes[3].imshow(np.ma.masked_where(~diff_mask, diff_map[..., 0]), cmap='gray') # Dummy for masking
        row_axes[3].imshow(diff_map, alpha=0.5)
        row_axes[3].set_title("Difference Map", fontsize=14)

        for ax in row_axes:
            ax.axis('off')

    plt.tight_layout()
    plt.subplots_adjust(top=0.92)

    # Save the file dynamically to the newly created folder
    # out_path = f'{test_output_folder}/Difference_Maps.png'
    plt.savefig(out_path, bbox_inches='tight', dpi=300)
    print(f"Saved visualization to: {out_path}")

    plt.show()

if not os.path.exists(pred_dir):
    print(f"Prediction directory not found: {pred_dir}")
else:
    # Fetch and sort the files
    all_files = sorted(os.listdir(pred_dir))

    # Categorize files based on original logic, keeping .npz and .nii.gz intact
    files_2d = [f for f in all_files if '2D' in f or f.endswith(('.png', '.jpg', '.jpeg', '.npz')) and '3D' not in f]
    files_3d = [f for f in all_files if '3D' in f or f.endswith('.nii.gz')]

    # Select the number of samples to output here
    SAMPLES_TO_VIEW = 10

    # Execute plotting
    print("Rendering 2D Modalities...")
    plot_comparisons(files_2d, "2D", num_samples=SAMPLES_TO_VIEW)

    print("Rendering 3D Modalities (Middle Slices)...")
    plot_comparisons(files_3d, "3D", num_samples=SAMPLES_TO_VIEW)

In [ ]:
#### plot worse score results ###


import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import SimpleITK as sitk


# --- Configuration ---
base_dir = '/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data'
dataset_name = 'kvasir_instruments' # Change to 'kvasir_seg' when you switch datasets

# Directories
img_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/imgs'
gt_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/gts'
pred_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/segs_og_ckpt'
csv_path = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/metrics_og_ckpt.csv'

# Define and create the missing output folder
test_output_folder = f'/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/experiments/{dataset_name}/og_ckpt'
out_path = f'{test_output_folder}/worse_10_ex.png'


# Helper function to load arrays safely
def get_array(path):
    if not os.path.exists(path): return None
    try:
        if path.endswith('.npz'):
            data = np.load(path)
            return data[data.files[0]]
        elif path.endswith(('.png', '.jpg', '.jpeg')):
            if 'imgs' in path or 'images' in path:
                return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
            else:
                return cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        elif path.endswith(('.nii.gz', '.nii')):
            return sitk.GetArrayFromImage(sitk.ReadImage(path))
    except Exception as e:
        print(f"Error loading {path}: {e}")
    return None

def plot_comparisons(file_list, category_name, metrics_dict):
    if not file_list:
        print(f"No files found for {category_name}.")
        return

    # Create a figure with 4 columns per modality
    fig, axes = plt.subplots(len(file_list), 4, figsize=(22, 5 * len(file_list)))

    # Handle single row indexing edge-case
    if len(file_list) == 1: axes = [axes]

    # Add a custom master legend for the colors
    tp_patch = mpatches.Patch(color='yellow', label='Match (True Positive)')
    fn_patch = mpatches.Patch(color='lime', label='Missed (False Negative)')
    fp_patch = mpatches.Patch(color='red', label='Over-predicted (False Positive)')
    fig.legend(handles=[tp_patch, fn_patch, fp_patch], loc='upper center', ncol=3, fontsize=14, bbox_to_anchor=(0.5, 0.98))
    fig.suptitle(f"{dataset_name} - {category_name} Modalities - Worst Results Analysis", fontsize=20, fontweight='bold', y=1.02)

    for i, filename in enumerate(file_list):
        img = get_array(os.path.join(img_dir, filename))
        gt = get_array(os.path.join(gt_dir, filename))
        pred = get_array(os.path.join(pred_dir, filename))

        if img is None or gt is None or pred is None:
            continue

        # Handle 3D slicing
        if img.ndim == 3 and img.shape[-1] != 3:
            mid = img.shape[0] // 2
            img, gt, pred = img[mid], gt[mid], pred[mid]

        # Standardize masks to binary (0 and 1)
        gt_bin = (gt > 0).astype(np.uint8)
        pred_bin = (pred > 0).astype(np.uint8)

        # Build the RGB Difference Map
        diff_map = np.zeros((*gt_bin.shape, 3), dtype=np.uint8)
        diff_map[(gt_bin == 1) & (pred_bin == 1)] = [255, 255, 0] # Yellow
        diff_map[(gt_bin == 1) & (pred_bin == 0)] = [0, 255, 0]   # Lime Green
        diff_map[(gt_bin == 0) & (pred_bin == 1)] = [255, 0, 0]   # Red

        # Clean title string
        clean_title = filename.replace('2DBox_', '').replace('3DBox_', '').replace('_demo.npz', '').replace('.nii.gz', '').replace('.npz', '')

        # Retrieve the scores for this specific image
        scores = metrics_dict.get(filename, {'dsc': 0.0, 'nsd': 0.0})
        score_text = f"DSC: {scores['dsc']:.4f} | NSD: {scores['nsd']:.4f}"

        row_axes = axes[i]

        # Col 1: Original
        row_axes[0].imshow(img, cmap='gray')
        row_axes[0].set_title(f"{clean_title}\nOriginal Image", fontsize=14)

        # Col 2: Ground Truth
        row_axes[1].imshow(img, cmap='gray')
        row_axes[1].imshow(np.ma.masked_where(gt_bin == 0, gt_bin), cmap='autumn', alpha=0.6)
        row_axes[1].set_title("Ground Truth", fontsize=14)

        # Col 3: Prediction (Now includes the scores)
        row_axes[2].imshow(img, cmap='gray')
        row_axes[2].imshow(np.ma.masked_where(pred_bin == 0, pred_bin), cmap='cool', alpha=0.6)
        row_axes[2].set_title(f"Model Prediction\n{score_text}", fontsize=14, color='darkred' if scores['dsc'] < 0.5 else 'black')

        # Col 4: Difference Map (Overlay)
        row_axes[3].imshow(img, cmap='gray')
        diff_mask = np.any(diff_map != [0, 0, 0], axis=-1)
        row_axes[3].imshow(np.ma.masked_where(~diff_mask, diff_map[..., 0]), cmap='gray')
        row_axes[3].imshow(diff_map, alpha=0.5)
        row_axes[3].set_title("Difference Map", fontsize=14)

        for ax in row_axes:
            ax.axis('off')

    plt.tight_layout()
    plt.subplots_adjust(top=0.92)

    # Save the file dynamically
    plt.savefig(out_path, bbox_inches='tight', dpi=300)
    print(f"Saved visualization to: {out_path}")

    plt.show()

# --- Main Execution Block ---
WORST_N_SAMPLES = 10 # Set how many of the worst cases you want to see

if not os.path.exists(pred_dir):
    print(f"Prediction directory not found: {pred_dir}")
elif not os.path.exists(csv_path):
    print(f"Metrics CSV not found at: {csv_path}")
else:
    # 1. Read the metrics CSV and sort by DSC to find the worst results
    df = pd.read_csv(csv_path)
    df_sorted = df.sort_values(by='dsc', ascending=True)
    worst_cases = df_sorted.head(WORST_N_SAMPLES)

    # 2. Map the worst cases to their actual filenames in the prediction directory
    available_preds = os.listdir(pred_dir)
    target_files = []
    metrics_map = {}

    for _, row in worst_cases.iterrows():
        case_id = str(row['case'])

        # Find the matching file in the prediction directory
        # (This handles the fact that the CSV case name might not have the file extension)
        matching_files = [f for f in available_preds if case_id in f]

        if matching_files:
            target_file = matching_files[0]
            target_files.append(target_file)
            metrics_map[target_file] = {'dsc': row['dsc'], 'nsd': row['nsd']}
        else:
            print(f"Warning: Could not find prediction file for case {case_id}")

    # 3. Categorize the fetched worst files based on original logic
    files_2d = [f for f in target_files if '2D' in f or f.endswith(('.png', '.jpg', '.jpeg', '.npz')) and '3D' not in f]
    files_3d = [f for f in target_files if '3D' in f or f.endswith('.nii.gz')]

    # 4. Execute plotting with the injected metrics dictionary
    if files_2d:
        print("Rendering Worst 2D Modalities...")
        plot_comparisons(files_2d, "2D", metrics_dict=metrics_map)

    if files_3d:
        print("Rendering Worst 3D Modalities...")
        plot_comparisons(files_3d, "3D", metrics_dict=metrics_map)

#Teeth Dataset

In [ ]:
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/

print("Extracting dental dataset...")
!unzip -q data/Dental_Caries_Segmentation.v3i.sam2.zip -d data/Dental_Caries_Segmentation/

# Final Cleanup
!rm data/Dental_Caries_Segmentation.v3i.sam2.zip
print("Dental dataset successfully added!")

In [ ]:
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/

In [ ]:
%%writefile preprocess_tooth_ds.py

import json
import os
import cv2
import numpy as np
import shutil
import argparse

def create_binary_dataset(coco_json_path, source_images_dir, output_dir):
    # Create the output folder structure
    images_out_dir = os.path.join(output_dir, "images")
    masks_out_dir = os.path.join(output_dir, "masks")
    os.makedirs(images_out_dir, exist_ok=True)
    os.makedirs(masks_out_dir, exist_ok=True)

    # Load the COCO JSON data
    with open(coco_json_path, 'r') as f:
        coco_data = json.load(f)

    # Target category IDs for the unified "disease" class
    # 1: Caries, 2: Cavity, 3: Crack. (4: Tooth is ignored and left as background)
    disease_category_ids = {1, 2, 3}

    # Group annotations by image ID for bulk processing
    annotations_by_image = {}
    for ann in coco_data.get('annotations', []):
        img_id = ann['image_id']
        if img_id not in annotations_by_image:
            annotations_by_image[img_id] = []
        annotations_by_image[img_id].append(ann)

    bboxes_json_data = {}

    # Process each image
    for img in coco_data.get('images', []):
        image_id = img['id']
        file_name = img['file_name']
        height = img['height']
        width = img['width']

        # Use filename without extension as the unique key
        file_key = os.path.splitext(file_name)[0]

        # Initialize a black 8-bit binary mask (background)
        mask = np.zeros((height, width), dtype=np.uint8)

        if image_id in annotations_by_image:
            for ann in annotations_by_image[image_id]:
                # Only draw the mask if it belongs to the disease classes
                if ann['category_id'] in disease_category_ids:
                    if 'segmentation' in ann and isinstance(ann['segmentation'], list):
                        for seg in ann['segmentation']:
                            # Reshape COCO polygon list into (x, y) coordinates
                            poly = np.array(seg, dtype=np.int32).reshape((-1, 2))
                            # Fill the polygon with white (255)
                            cv2.fillPoly(mask, [poly], 255)

        # Generate new, non-overlapping bounding boxes from the unified mask
        # RETR_EXTERNAL ensures we only get the outermost boundaries, merging overlaps
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        merged_bboxes = []
        for contour in contours:
            # Filter out microscopic noise artifacts (threshold set to 5 pixels)
            if cv2.contourArea(contour) > 5:
                x, y, w, h = cv2.boundingRect(contour)
                merged_bboxes.append({
                    "label": "disease",
                    "xmin": int(x),
                    "ymin": int(y),
                    "xmax": int(x + w),
                    "ymax": int(y + h)
                })

        # Construct the JSON structure for this specific image
        bboxes_json_data[file_key] = {
            "height": height,
            "width": width,
            "bbox": merged_bboxes
        }

        # Save the binary mask as PNG
        mask_filename = f"{file_key}.png"
        cv2.imwrite(os.path.join(masks_out_dir, mask_filename), mask)

        # Route the original image
        src_img_path = os.path.join(source_images_dir, file_name)
        dst_img_path = os.path.join(images_out_dir, f"{file_key}.jpg")

        if os.path.exists(src_img_path):
            shutil.copy(src_img_path, dst_img_path)
        else:
            print(f"Warning: Image {file_name} not found in {source_images_dir}")

    # Save the central bboxes.json to the root of the output directory
    with open(os.path.join(output_dir, "bboxes.json"), 'w') as f:
        json.dump(bboxes_json_data, f)

    print(f"Binary dataset generated. Saved to '{output_dir}'.")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Convert COCO JSON to a binary mask dataset format.")
    parser.add_argument("--json_path", type=str, required=True, help="Path to the COCO _annotations.coco.json file")
    parser.add_argument("--img_dir", type=str, required=True, help="Directory containing the original JPG images")
    parser.add_argument("--out_dir", type=str, required=True, help="Target output directory for the generated dataset")

    args = parser.parse_args()

    create_binary_dataset(
        coco_json_path=args.json_path,
        source_images_dir=args.img_dir,
        output_dir=args.out_dir
    )

In [ ]:
### preprocess the multi lable tooth dataset into a binary dataset with folders matching the lite med sam ###
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/


!python preprocess_tooth_ds.py \
  --json_path "data/dental_caries_segmentation/train/_annotations.coco.json" \
  --img_dir "data/dental_caries_segmentation/train" \
  --out_dir "data/dental_caries_segmentation_processed/train/"

In [ ]:
### preprocess the multi lable tooth dataset into a binary dataset with folders matching the lite med sam ###
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/


!python preprocess_tooth_ds.py \
  --json_path "data/dental_caries_segmentation/test/_annotations.coco.json" \
  --img_dir "data/dental_caries_segmentation/test" \
  --out_dir "data/dental_caries_segmentation_processed/test/"

In [ ]:
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/

## preprocess Tooth DS

!python MedSAM/pre_rgb_bbox_datasets.py \
  -output_path data/dental_caries_segmentation_processed \
  -modality RGB \
  -anatomy Teeth \
  -img_name_suffix .jpg \
  -gt_name_suffix .png \
  -bbox_format json \
  -train_img_path data/dental_caries_segmentation_processed/train/images \
  -train_gt_path data/dental_caries_segmentation_processed/train/masks \
  -train_bbox_json data/dental_caries_segmentation_processed/train/bboxes.json \
  -test_img_path data/dental_caries_segmentation_processed/test/images \
  -test_gt_path data/dental_caries_segmentation_processed/test/masks \
  -test_bbox_json data/dental_caries_segmentation_processed/test/bboxes.json \
  --binary_mask

In [ ]:
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/
import numpy as np

file_path = "data/npz/MedSAM_train/RGB_Dataset/imgs/RGB_Dataset_1578_jpg.rf.19e73a9bb4ce3dc94a5686c1006051df_2.npz"

try:
    data = np.load(file_path)
    print("Files successfully found inside the archive:", data.files)
    print("Image shape:", data['imgs'].shape)
    print("Boxes shape:", data['boxes'].shape)
except Exception as e:
    print("The file is corrupted:", e)

In [ ]:
### inference on Teeth ds test on original lite medsam checkpoint ###

%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
# Reccomanded Sanity Test
!python CVPR24_LiteMedSAM_infer.py -i /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_test/RGB_Teeth/imgs/ -o /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_test/RGB_Teeth/segs_og_ckpt

In [ ]:
### metric compute on teeth ds test on original lite medsam checkpoint ###

%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/
# Compute Metrics
!python MedSAM/evaluation/compute_metrics.py -s data/dental_caries_segmentation_processed/MedSAM_test/RGB_Teeth/segs_og_ckpt -g data/dental_caries_segmentation_processed/MedSAM_test/RGB_Teeth/gts -csv_dir data/dental_caries_segmentation_processed/MedSAM_test/RGB_Teeth/metrics_og_ckpt.csv


In [ ]:
### metrics distribution of teeth ds on original lite med sam checkpoint

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the path to your metrics.csv file
csv_path = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_test/RGB_Teeth/metrics_og_ckpt.csv"

# Load the dataset
df = pd.read_csv(csv_path)

# 1. Calculate and print the requested statistics
metrics = ['dsc', 'nsd']
print("--- Inference Statistics ---")
for metric in metrics:
    avg_val = df[metric].mean()
    med_val = df[metric].median()
    std_val = df[metric].std()

    print(f"\nMetric: {metric.upper()}")
    print(f"  Average (Mean): {avg_val:.4f}")
    print(f"  Median:         {med_val:.4f}")
    print(f"  Std Dev:        {std_val:.4f}")

# 2. Create the visualizations
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, metric in enumerate(metrics):
    # Boxplot shows the median, quartiles, and identifies outliers
    sns.boxplot(
        y=df[metric],
        ax=axes[i],
        color="lightgray",
        width=0.4,
        showfliers=False # Hide default outliers to prevent double-plotting with stripplot
    )

    # Stripplot overlays the actual distribution of all data points
    sns.stripplot(
        y=df[metric],
        ax=axes[i],
        color=sns.color_palette("husl", 2)[i],
        alpha=0.6,
        jitter=True,
        size=5
    )

    axes[i].set_title(f"Distribution of {metric.upper()}", fontsize=14, fontweight='bold')
    axes[i].set_ylabel("Score", fontsize=12)
    axes[i].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

In [ ]:


# # --- Configuration Paths ---
# base_dir = '/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data'
# dataset_name = 'kvasir_instruments' # Change to 'kvasir_seg' when you switch datasets

# img_dir  = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/imgs'
# gt_dir   = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/gts'
# pred_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/segs_og_ckpt'
# out_file = f'/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/experiments/{dataset_name}/og_ckpt/Difference_Maps.png'

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import SimpleITK as sitk
import random

# --- Configuration ---
base_dir = '/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data'
dataset_name = 'dental_caries_segmentation_processed' # Change to 'kvasir_seg' when you switch datasets

# Directories
img_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/imgs'
gt_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/gts'
pred_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/segs_og_ckpt'

# Define and create the missing output folder
test_output_folder = f'/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/experiments/{dataset_name}/og_ckpt'
out_path = f'{test_output_folder}/Difference_Maps_10_ex1.png'
# os.makedirs(f'{test_output_folder}', exist_ok=True)

# Helper function to load arrays safely
def get_array(path):
    if not os.path.exists(path): return None
    try:
        if path.endswith('.npz'):
            data = np.load(path)
            return data[data.files[0]]
        elif path.endswith(('.png', '.jpg', '.jpeg')):
            if 'imgs' in path or 'images' in path:
                return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
            else:
                return cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        elif path.endswith(('.nii.gz', '.nii')):
            return sitk.GetArrayFromImage(sitk.ReadImage(path))
    except Exception as e:
        print(f"Error loading {path}: {e}")
    return None

def plot_comparisons(file_list, category_name, num_samples=None):
    if not file_list:
        print(f"No files found for {category_name}.")
        return

    # Random sampling to limit output
    if num_samples and len(file_list) > num_samples:
        print(f"Sampling {num_samples} files out of {len(file_list)}...")
        file_list = random.sample(file_list, num_samples)

    # Create a figure with 4 columns per modality
    fig, axes = plt.subplots(len(file_list), 4, figsize=(22, 5 * len(file_list)))

    # Handle single row indexing edge-case
    if len(file_list) == 1: axes = [axes]

    # Add a custom master legend for the colors
    tp_patch = mpatches.Patch(color='yellow', label='Match (True Positive)')
    fn_patch = mpatches.Patch(color='lime', label='Missed (False Negative)')
    fp_patch = mpatches.Patch(color='red', label='Over-predicted (False Positive)')
    fig.legend(handles=[tp_patch, fn_patch, fp_patch], loc='upper center', ncol=3, fontsize=14, bbox_to_anchor=(0.5, 0.98))
    fig.suptitle(f"{dataset_name} - {category_name} Modalities - Ground Truth vs Prediction", fontsize=20, fontweight='bold', y=1.02)

    for i, filename in enumerate(file_list):
        img = get_array(os.path.join(img_dir, filename))
        gt = get_array(os.path.join(gt_dir, filename))
        pred = get_array(os.path.join(pred_dir, filename))

        if img is None or gt is None or pred is None:
            continue

        # Handle 3D slicing (Distinguish from 2D RGB images by checking the last channel)
        if img.ndim == 3 and img.shape[-1] != 3:
            mid = img.shape[0] // 2
            img, gt, pred = img[mid], gt[mid], pred[mid]

        # Standardize masks to binary (0 and 1)
        gt_bin = (gt > 0).astype(np.uint8)
        pred_bin = (pred > 0).astype(np.uint8)

        # Build the RGB Difference Map
        diff_map = np.zeros((*gt_bin.shape, 3), dtype=np.uint8)
        diff_map[(gt_bin == 1) & (pred_bin == 1)] = [255, 255, 0] # Yellow
        diff_map[(gt_bin == 1) & (pred_bin == 0)] = [0, 255, 0]   # Lime Green
        diff_map[(gt_bin == 0) & (pred_bin == 1)] = [255, 0, 0]   # Red

        # Clean title string
        clean_title = filename.replace('2DBox_', '').replace('3DBox_', '').replace('_demo.npz', '').replace('.nii.gz', '').replace('.npz', '')
        row_axes = axes[i]

        # Col 1: Original
        row_axes[0].imshow(img, cmap='gray')
        row_axes[0].set_title(f"{clean_title}\nOriginal Image", fontsize=14)

        # Col 2: Ground Truth
        row_axes[1].imshow(img, cmap='gray')
        # Using a custom colormap overlay to visualize GT
        row_axes[1].imshow(np.ma.masked_where(gt_bin == 0, gt_bin), cmap='autumn', alpha=0.6)
        row_axes[1].set_title("Ground Truth", fontsize=14)

        # Col 3: Prediction
        row_axes[2].imshow(img, cmap='gray')
        row_axes[2].imshow(np.ma.masked_where(pred_bin == 0, pred_bin), cmap='cool', alpha=0.6)
        row_axes[2].set_title("Model Prediction", fontsize=14)

        # Col 4: Difference Map (Overlay)
        row_axes[3].imshow(img, cmap='gray')
        # Mask out the background [0,0,0] so we only overlay the active errors/matches
        diff_mask = np.any(diff_map != [0, 0, 0], axis=-1)
        row_axes[3].imshow(np.ma.masked_where(~diff_mask, diff_map[..., 0]), cmap='gray') # Dummy for masking
        row_axes[3].imshow(diff_map, alpha=0.5)
        row_axes[3].set_title("Difference Map", fontsize=14)

        for ax in row_axes:
            ax.axis('off')

    plt.tight_layout()
    plt.subplots_adjust(top=0.92)

    # Save the file dynamically to the newly created folder
    # out_path = f'{test_output_folder}/Difference_Maps.png'
    plt.savefig(out_path, bbox_inches='tight', dpi=300)
    print(f"Saved visualization to: {out_path}")

    plt.show()

if not os.path.exists(pred_dir):
    print(f"Prediction directory not found: {pred_dir}")
else:
    # Fetch and sort the files
    all_files = sorted(os.listdir(pred_dir))

    # Categorize files based on original logic, keeping .npz and .nii.gz intact
    files_2d = [f for f in all_files if '2D' in f or f.endswith(('.png', '.jpg', '.jpeg', '.npz')) and '3D' not in f]
    files_3d = [f for f in all_files if '3D' in f or f.endswith('.nii.gz')]

    # Select the number of samples to output here
    SAMPLES_TO_VIEW = 10

    # Execute plotting
    print("Rendering 2D Modalities...")
    plot_comparisons(files_2d, "2D", num_samples=SAMPLES_TO_VIEW)

    print("Rendering 3D Modalities (Middle Slices)...")
    plot_comparisons(files_3d, "3D", num_samples=SAMPLES_TO_VIEW)

In [ ]:
#### plot worse score results ###


import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import SimpleITK as sitk


# --- Configuration ---
base_dir = '/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data'
dataset_name = 'dental_caries_segmentation_processed' # Change to 'kvasir_seg' when you switch datasets

# Directories
img_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/imgs'
gt_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/gts'
pred_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/segs_og_ckpt'
csv_path = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/metrics_og_ckpt.csv'

# Define and create the missing output folder
test_output_folder = f'/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/experiments/{dataset_name}/og_ckpt'
out_path = f'{test_output_folder}/worse_10_ex.png'


# Helper function to load arrays safely
def get_array(path):
    if not os.path.exists(path): return None
    try:
        if path.endswith('.npz'):
            data = np.load(path)
            return data[data.files[0]]
        elif path.endswith(('.png', '.jpg', '.jpeg')):
            if 'imgs' in path or 'images' in path:
                return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
            else:
                return cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        elif path.endswith(('.nii.gz', '.nii')):
            return sitk.GetArrayFromImage(sitk.ReadImage(path))
    except Exception as e:
        print(f"Error loading {path}: {e}")
    return None

def plot_comparisons(file_list, category_name, metrics_dict):
    if not file_list:
        print(f"No files found for {category_name}.")
        return

    # Create a figure with 4 columns per modality
    fig, axes = plt.subplots(len(file_list), 4, figsize=(22, 5 * len(file_list)))

    # Handle single row indexing edge-case
    if len(file_list) == 1: axes = [axes]

    # Add a custom master legend for the colors
    tp_patch = mpatches.Patch(color='yellow', label='Match (True Positive)')
    fn_patch = mpatches.Patch(color='lime', label='Missed (False Negative)')
    fp_patch = mpatches.Patch(color='red', label='Over-predicted (False Positive)')
    fig.legend(handles=[tp_patch, fn_patch, fp_patch], loc='upper center', ncol=3, fontsize=14, bbox_to_anchor=(0.5, 0.98))
    fig.suptitle(f"{dataset_name} - {category_name} Modalities - Worst Results Analysis", fontsize=20, fontweight='bold', y=1.02)

    for i, filename in enumerate(file_list):
        img = get_array(os.path.join(img_dir, filename))
        gt = get_array(os.path.join(gt_dir, filename))
        pred = get_array(os.path.join(pred_dir, filename))

        if img is None or gt is None or pred is None:
            continue

        # Handle 3D slicing
        if img.ndim == 3 and img.shape[-1] != 3:
            mid = img.shape[0] // 2
            img, gt, pred = img[mid], gt[mid], pred[mid]

        # Standardize masks to binary (0 and 1)
        gt_bin = (gt > 0).astype(np.uint8)
        pred_bin = (pred > 0).astype(np.uint8)

        # Build the RGB Difference Map
        diff_map = np.zeros((*gt_bin.shape, 3), dtype=np.uint8)
        diff_map[(gt_bin == 1) & (pred_bin == 1)] = [255, 255, 0] # Yellow
        diff_map[(gt_bin == 1) & (pred_bin == 0)] = [0, 255, 0]   # Lime Green
        diff_map[(gt_bin == 0) & (pred_bin == 1)] = [255, 0, 0]   # Red

        # Clean title string
        clean_title = filename.replace('2DBox_', '').replace('3DBox_', '').replace('_demo.npz', '').replace('.nii.gz', '').replace('.npz', '')

        # Retrieve the scores for this specific image
        scores = metrics_dict.get(filename, {'dsc': 0.0, 'nsd': 0.0})
        score_text = f"DSC: {scores['dsc']:.4f} | NSD: {scores['nsd']:.4f}"

        row_axes = axes[i]

        # Col 1: Original
        row_axes[0].imshow(img, cmap='gray')
        row_axes[0].set_title(f"{clean_title}\nOriginal Image", fontsize=14)

        # Col 2: Ground Truth
        row_axes[1].imshow(img, cmap='gray')
        row_axes[1].imshow(np.ma.masked_where(gt_bin == 0, gt_bin), cmap='autumn', alpha=0.6)
        row_axes[1].set_title("Ground Truth", fontsize=14)

        # Col 3: Prediction (Now includes the scores)
        row_axes[2].imshow(img, cmap='gray')
        row_axes[2].imshow(np.ma.masked_where(pred_bin == 0, pred_bin), cmap='cool', alpha=0.6)
        row_axes[2].set_title(f"Model Prediction\n{score_text}", fontsize=14, color='darkred' if scores['dsc'] < 0.5 else 'black')

        # Col 4: Difference Map (Overlay)
        row_axes[3].imshow(img, cmap='gray')
        diff_mask = np.any(diff_map != [0, 0, 0], axis=-1)
        row_axes[3].imshow(np.ma.masked_where(~diff_mask, diff_map[..., 0]), cmap='gray')
        row_axes[3].imshow(diff_map, alpha=0.5)
        row_axes[3].set_title("Difference Map", fontsize=14)

        for ax in row_axes:
            ax.axis('off')

    plt.tight_layout()
    plt.subplots_adjust(top=0.92)

    # Save the file dynamically
    plt.savefig(out_path, bbox_inches='tight', dpi=300)
    print(f"Saved visualization to: {out_path}")

    plt.show()

# --- Main Execution Block ---
WORST_N_SAMPLES = 10 # Set how many of the worst cases you want to see

if not os.path.exists(pred_dir):
    print(f"Prediction directory not found: {pred_dir}")
elif not os.path.exists(csv_path):
    print(f"Metrics CSV not found at: {csv_path}")
else:
    # 1. Read the metrics CSV and sort by DSC to find the worst results
    df = pd.read_csv(csv_path)
    df_sorted = df.sort_values(by='dsc', ascending=True)
    worst_cases = df_sorted.head(WORST_N_SAMPLES)

    # 2. Map the worst cases to their actual filenames in the prediction directory
    available_preds = os.listdir(pred_dir)
    target_files = []
    metrics_map = {}

    for _, row in worst_cases.iterrows():
        case_id = str(row['case'])

        # Find the matching file in the prediction directory
        # (This handles the fact that the CSV case name might not have the file extension)
        matching_files = [f for f in available_preds if case_id in f]

        if matching_files:
            target_file = matching_files[0]
            target_files.append(target_file)
            metrics_map[target_file] = {'dsc': row['dsc'], 'nsd': row['nsd']}
        else:
            print(f"Warning: Could not find prediction file for case {case_id}")

    # 3. Categorize the fetched worst files based on original logic
    files_2d = [f for f in target_files if '2D' in f or f.endswith(('.png', '.jpg', '.jpeg', '.npz')) and '3D' not in f]
    files_3d = [f for f in target_files if '3D' in f or f.endswith('.nii.gz')]

    # 4. Execute plotting with the injected metrics dictionary
    if files_2d:
        print("Rendering Worst 2D Modalities...")
        plot_comparisons(files_2d, "2D", metrics_dict=metrics_map)

    if files_3d:
        print("Rendering Worst 3D Modalities...")
        plot_comparisons(files_3d, "3D", metrics_dict=metrics_map)

##Debugging problem with inference on teeth ds

In [ ]:
" part of debug dont repeat "
import glob, os
from PIL import Image
R = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed"
sizes = []
for split in ["train", "test"]:
    for f in glob.glob(f"{R}/{split}/images/*.jpg"):
        w, h = Image.open(f).size
        sizes.append((w * h / 1e6, w, h, split, os.path.basename(f)))
sizes.sort(reverse=True)
for mp, w, h, s, n in sizes[:10]: print(f"{mp:5.1f} MP  {w}x{h}  {s}  {n}")
print("median MP:", sizes[len(sizes) // 2][0])

In [ ]:
" part of debug dont repeat "

size_of = {n[:-4]: (mp, w, h) for mp, w, h, s, n in sizes}
npz = sorted(os.path.basename(f) for f in glob.glob(f"{R}/MedSAM_test/RGB_Teeth/imgs/*.npz"))
for i, f in enumerate(npz[22:30], start=23):
    mp, w, h = size_of[f[len("RGB_Teeth_"):-4]]
    print(f"{i:3d}  {mp:5.1f} MP  {w}x{h}  {f}")
print("longest side > 1024:", sum(max(w, h) > 1024 for _, w, h, _, _ in sizes), "of", len(sizes))

In [ ]:
" part of debug dont repeat "

size_of = {n[:-4]: (mp, w, h) for mp, w, h, s, n in sizes}
npz = sorted(os.path.basename(f) for f in glob.glob(f"{R}/MedSAM_test/RGB_Teeth/imgs/*.npz"))
for i, f in enumerate(npz[:30], start=1):
    mp, w, h = size_of[f[len("RGB_Teeth_"):-4]]
    print(f"{i:3d}  {mp:5.1f} MP  {w}x{h}  {f}")
print("longest side > 1024:", sum(max(w, h) > 1024 for _, w, h, _, _ in sizes), "of", len(sizes))

Checking again if something is corrupted with image 1578

In [ ]:
" part of debug dont repeat "

import os, shutil, time, numpy as np
T = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_test/RGB_Teeth"
f = f"{T}/imgs/RGB_Teeth_1578_jpg.rf.19e73a9bb4ce3dc94a5686c1006051df.npz"
t = time.time(); a = np.load(f)["imgs"]; print("read from Drive:", round(time.time() - t, 1), "s", a.shape)
os.makedirs("/content/one", exist_ok=True); shutil.copy(f, "/content/one/")

In [ ]:
" part of debug dont repeat "

%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
!python CVPR24_LiteMedSAM_infer.py -i /content/one/ -o /content/one_out; echo "exit code: $?"

In [ ]:
" part of debug dont repeat "


# Estimating the peak memory for every test image
import glob, os, numpy as np
from PIL import Image
R = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed"
rows = []
for f in sorted(glob.glob(f"{R}/MedSAM_test/RGB_Teeth/imgs/*.npz")):
    stem = os.path.basename(f)[len("RGB_Teeth_"):-4]
    w, h = Image.open(f"{R}/test/images/{stem}.jpg").size
    nb, mp = len(np.load(f)["boxes"]), w * h / 1e6
    rows.append((0.15 + mp * (0.04 + 0.032 * nb), mp, nb, stem))
rows.sort(reverse=True)
for gb, mp, nb, s in rows[:10]: print(f"~{gb:4.1f} GB  {mp:5.1f} MP  {nb:2d} boxes  {s}")

In [ ]:
# Turn off the overlay by default to deal with suspected RAM issue
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
!sed -i "/'--save_overlay'/,/help=/ s/default=True/default=False/" CVPR24_LiteMedSAM_infer.py
!grep -n -A2 "'--save_overlay'" CVPR24_LiteMedSAM_infer.py

In [ ]:
# Inference on Teeth ds test on original lite medsam checkpoint (turned off overlay)
T = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_test/RGB_Teeth"
!python CVPR24_LiteMedSAM_infer.py -i {T}/imgs/ -o {T}/segs_og_ckpt; echo "exit code: $?"

In [ ]:
" since compute matrix doesn't fit many bounding boxes: "
# metric computation on teeth ds test on original lite medsam checkpoint
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/
# Compute Metrics
!python MedSAM/evaluation/compute_metrics.py -s {T}/segs_og_ckpt -g {T}/gts -csv_dir {T}/metrics_og_ckpt.csv

In [ ]:
" stats with incorrect bounding boxes understanding "
# metrics distribution of teeth ds on original lite med sam checkpoint

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the path to our metrics.csv file
csv_path = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_test/RGB_Teeth/metrics_og_ckpt.csv"

df = pd.read_csv(csv_path) # Load the dataset

# Calculate and print statistics
metrics = ['dsc', 'nsd']
print("--- Inference Statistics ---")
for metric in metrics:
    avg_val = df[metric].mean()
    med_val = df[metric].median()
    std_val = df[metric].std()

    print(f"\nMetric: {metric.upper()}")
    print(f"  Average (Mean): {avg_val:.4f}")
    print(f"  Median:         {med_val:.4f}")
    print(f"  Std Dev:        {std_val:.4f}")

# Create visualizations
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, metric in enumerate(metrics):
    # Boxplot shows the median, quartiles, and identifies outliers
    sns.boxplot(
        y=df[metric],
        ax=axes[i],
        color="lightgray",
        width=0.4,
        showfliers=False # Hides default outliers to prevent double-plotting with stripplot
    )

    # Stripplot overlays the actual distribution of all data points
    sns.stripplot(
        y=df[metric],
        ax=axes[i],
        color=sns.color_palette("husl", 2)[i],
        alpha=0.6,
        jitter=True,
        size=5
    )

    axes[i].set_title(f"Distribution of {metric.upper()}", fontsize=14, fontweight='bold')
    axes[i].set_ylabel("Score", fontsize=12)
    axes[i].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# # --- Configuration Paths ---
# base_dir = '/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data'
# dataset_name = 'kvasir_instruments' # Change to 'kvasir_seg' when you switch datasets

# img_dir  = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/imgs'
# gt_dir   = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/gts'
# pred_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_KvasirInstruments/segs_og_ckpt'
# out_file = f'/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/experiments/{dataset_name}/og_ckpt/Difference_Maps.png'

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import SimpleITK as sitk
import random

# --- Configuration ---
base_dir = '/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data'
dataset_name = 'dental_caries_segmentation_processed'

# Directories
img_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/imgs'
gt_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/gts'
pred_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/segs_og_ckpt'

# Define and create the missing output folder
test_output_folder = f'/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/experiments/{dataset_name}/og_ckpt'
out_path = f'{test_output_folder}/Difference_Maps_10_ds_teeth.png'
os.makedirs(f'{test_output_folder}', exist_ok=True)

# Helper function to load arrays safely
def get_array(path):
    if not os.path.exists(path): return None
    try:
        if path.endswith('.npz'):
            data = np.load(path)
            return data[data.files[0]]
        elif path.endswith(('.png', '.jpg', '.jpeg')):
            if 'imgs' in path or 'images' in path:
                return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
            else:
                return cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        elif path.endswith(('.nii.gz', '.nii')):
            return sitk.GetArrayFromImage(sitk.ReadImage(path))
    except Exception as e:
        print(f"Error loading {path}: {e}")
    return None

def plot_comparisons(file_list, category_name, num_samples=None):
    if not file_list:
        print(f"No files found for {category_name}.")
        return

    # Random sampling to limit output
    if num_samples and len(file_list) > num_samples:
        print(f"Sampling {num_samples} files out of {len(file_list)}...")
        file_list = random.sample(file_list, num_samples)

    # Create a figure with 4 columns per modality
    fig, axes = plt.subplots(len(file_list), 4, figsize=(22, 5 * len(file_list)))

    # Handle single row indexing edge-case
    if len(file_list) == 1: axes = [axes]

    # Add a custom master legend for the colors
    tp_patch = mpatches.Patch(color='yellow', label='Match (True Positive)')
    fn_patch = mpatches.Patch(color='lime', label='Missed (False Negative)')
    fp_patch = mpatches.Patch(color='red', label='Over-predicted (False Positive)')
    fig.legend(handles=[tp_patch, fn_patch, fp_patch], loc='upper center', ncol=3, fontsize=14, bbox_to_anchor=(0.5, 0.98))
    fig.suptitle(f"{dataset_name} - {category_name} Modalities - Ground Truth vs Prediction", fontsize=20, fontweight='bold', y=1.02)

    for i, filename in enumerate(file_list):
        img = get_array(os.path.join(img_dir, filename))
        gt = get_array(os.path.join(gt_dir, filename))
        pred = get_array(os.path.join(pred_dir, filename))

        if img is None or gt is None or pred is None:
            continue

        # Handle 3D slicing (Distinguish from 2D RGB images by checking the last channel)
        if img.ndim == 3 and img.shape[-1] != 3:
            mid = img.shape[0] // 2
            img, gt, pred = img[mid], gt[mid], pred[mid]

        # Standardize masks to binary (0 and 1)
        gt_bin = (gt > 0).astype(np.uint8)
        pred_bin = (pred > 0).astype(np.uint8)

        # Build the RGB Difference Map
        diff_map = np.zeros((*gt_bin.shape, 3), dtype=np.uint8)
        diff_map[(gt_bin == 1) & (pred_bin == 1)] = [255, 255, 0] # Yellow
        diff_map[(gt_bin == 1) & (pred_bin == 0)] = [0, 255, 0]   # Lime Green
        diff_map[(gt_bin == 0) & (pred_bin == 1)] = [255, 0, 0]   # Red

        # Clean title string
        clean_title = filename.replace('2DBox_', '').replace('3DBox_', '').replace('_demo.npz', '').replace('.nii.gz', '').replace('.npz', '')
        row_axes = axes[i]

        # Col 1: Original
        row_axes[0].imshow(img, cmap='gray')
        row_axes[0].set_title(f"{clean_title}\nOriginal Image", fontsize=14)

        # Col 2: Ground Truth
        row_axes[1].imshow(img, cmap='gray')
        # Using a custom colormap overlay to visualize GT
        row_axes[1].imshow(np.ma.masked_where(gt_bin == 0, gt_bin), cmap='autumn', alpha=0.6)
        row_axes[1].set_title("Ground Truth", fontsize=14)

        # Col 3: Prediction
        row_axes[2].imshow(img, cmap='gray')
        row_axes[2].imshow(np.ma.masked_where(pred_bin == 0, pred_bin), cmap='cool', alpha=0.6)
        row_axes[2].set_title("Model Prediction", fontsize=14)

        # Col 4: Difference Map (Overlay)
        row_axes[3].imshow(img, cmap='gray')
        # Mask out the background [0,0,0] so we only overlay the active errors/matches
        diff_mask = np.any(diff_map != [0, 0, 0], axis=-1)
        row_axes[3].imshow(np.ma.masked_where(~diff_mask, diff_map[..., 0]), cmap='gray') # Dummy for masking
        row_axes[3].imshow(diff_map, alpha=0.5)
        row_axes[3].set_title("Difference Map", fontsize=14)

        for ax in row_axes:
            ax.axis('off')

    plt.tight_layout()
    plt.subplots_adjust(top=0.92)

    # Save the file dynamically to the newly created folder
    # out_path = f'{test_output_folder}/Difference_Maps.png'
    plt.savefig(out_path, bbox_inches='tight', dpi=300)
    print(f"Saved visualization to: {out_path}")

    plt.show()

if not os.path.exists(pred_dir):
    print(f"Prediction directory not found: {pred_dir}")
else:
    # Fetch and sort the files
    all_files = sorted(os.listdir(pred_dir))

    # Categorize files based on original logic, keeping .npz and .nii.gz intact
    files_2d = [f for f in all_files if '2D' in f or f.endswith(('.png', '.jpg', '.jpeg', '.npz')) and '3D' not in f]
    files_3d = [f for f in all_files if '3D' in f or f.endswith('.nii.gz')]

    # Select the number of samples to output here
    SAMPLES_TO_VIEW = 10

    # Execute plotting
    print("Rendering 2D Modalities...")
    plot_comparisons(files_2d, "2D", num_samples=SAMPLES_TO_VIEW)

    print("Rendering 3D Modalities (Middle Slices)...")
    plot_comparisons(files_3d, "3D", num_samples=SAMPLES_TO_VIEW)

In [ ]:
# Plot worst score results
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import SimpleITK as sitk

# --- Configuration ---
base_dir = '/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data'
dataset_name = 'dental_caries_segmentation_processed'

# Directories
img_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/imgs'
gt_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/gts'
pred_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/segs_og_ckpt'
csv_path = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/metrics_og_ckpt.csv'

# Define and create the missing output folder
test_output_folder = f'/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/experiments/{dataset_name}/og_ckpt'
out_path = f'{test_output_folder}/worst_10_ds_teeth.png'


# Helper function to load arrays safely
def get_array(path):
    if not os.path.exists(path): return None
    try:
        if path.endswith('.npz'):
            data = np.load(path)
            return data[data.files[0]]
        elif path.endswith(('.png', '.jpg', '.jpeg')):
            if 'imgs' in path or 'images' in path:
                return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
            else:
                return cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        elif path.endswith(('.nii.gz', '.nii')):
            return sitk.GetArrayFromImage(sitk.ReadImage(path))
    except Exception as e:
        print(f"Error loading {path}: {e}")
    return None

def plot_comparisons(file_list, category_name, metrics_dict):
    if not file_list:
        print(f"No files found for {category_name}.")
        return

    # Create a figure with 4 columns per modality
    fig, axes = plt.subplots(len(file_list), 4, figsize=(22, 5 * len(file_list)))

    # Handle single row indexing edge-case
    if len(file_list) == 1: axes = [axes]

    # Add a custom master legend for the colors
    tp_patch = mpatches.Patch(color='yellow', label='Match (True Positive)')
    fn_patch = mpatches.Patch(color='lime', label='Missed (False Negative)')
    fp_patch = mpatches.Patch(color='red', label='Over-predicted (False Positive)')
    fig.legend(handles=[tp_patch, fn_patch, fp_patch], loc='upper center', ncol=3, fontsize=14, bbox_to_anchor=(0.5, 0.98))
    fig.suptitle(f"{dataset_name} - {category_name} Modalities - Worst Results Analysis", fontsize=20, fontweight='bold', y=1.02)

    for i, filename in enumerate(file_list):
        img = get_array(os.path.join(img_dir, filename))
        gt = get_array(os.path.join(gt_dir, filename))
        pred = get_array(os.path.join(pred_dir, filename))

        if img is None or gt is None or pred is None:
            continue

        # Handle 3D slicing
        if img.ndim == 3 and img.shape[-1] != 3:
            mid = img.shape[0] // 2
            img, gt, pred = img[mid], gt[mid], pred[mid]

        # Standardize masks to binary (0 and 1)
        gt_bin = (gt > 0).astype(np.uint8)
        pred_bin = (pred > 0).astype(np.uint8)

        # Build the RGB Difference Map
        diff_map = np.zeros((*gt_bin.shape, 3), dtype=np.uint8)
        diff_map[(gt_bin == 1) & (pred_bin == 1)] = [255, 255, 0] # Yellow
        diff_map[(gt_bin == 1) & (pred_bin == 0)] = [0, 255, 0]   # Lime Green
        diff_map[(gt_bin == 0) & (pred_bin == 1)] = [255, 0, 0]   # Red

        # Clean title string
        clean_title = filename.replace('2DBox_', '').replace('3DBox_', '').replace('_demo.npz', '').replace('.nii.gz', '').replace('.npz', '')

        # Retrieve the scores for this specific image
        scores = metrics_dict.get(filename, {'dsc': 0.0, 'nsd': 0.0})
        score_text = f"DSC: {scores['dsc']:.4f} | NSD: {scores['nsd']:.4f}"

        row_axes = axes[i]

        # Col 1: Original
        row_axes[0].imshow(img, cmap='gray')
        row_axes[0].set_title(f"{clean_title}\nOriginal Image", fontsize=14)

        # Col 2: Ground Truth
        row_axes[1].imshow(img, cmap='gray')
        row_axes[1].imshow(np.ma.masked_where(gt_bin == 0, gt_bin), cmap='autumn', alpha=0.6)
        row_axes[1].set_title("Ground Truth", fontsize=14)

        # Col 3: Prediction (Now includes the scores)
        row_axes[2].imshow(img, cmap='gray')
        row_axes[2].imshow(np.ma.masked_where(pred_bin == 0, pred_bin), cmap='cool', alpha=0.6)
        row_axes[2].set_title(f"Model Prediction\n{score_text}", fontsize=14, color='darkred' if scores['dsc'] < 0.5 else 'black')

        # Col 4: Difference Map (Overlay)
        row_axes[3].imshow(img, cmap='gray')
        diff_mask = np.any(diff_map != [0, 0, 0], axis=-1)
        row_axes[3].imshow(np.ma.masked_where(~diff_mask, diff_map[..., 0]), cmap='gray')
        row_axes[3].imshow(diff_map, alpha=0.5)
        row_axes[3].set_title("Difference Map", fontsize=14)

        for ax in row_axes:
            ax.axis('off')

    plt.tight_layout()
    plt.subplots_adjust(top=0.92)

    # Save the file dynamically
    plt.savefig(out_path, bbox_inches='tight', dpi=300)
    print(f"Saved visualization to: {out_path}")

    plt.show()

# --- Main Execution Block ---
WORST_N_SAMPLES = 10 # Set how many of the worst cases you want to see

if not os.path.exists(pred_dir):
    print(f"Prediction directory not found: {pred_dir}")
elif not os.path.exists(csv_path):
    print(f"Metrics CSV not found at: {csv_path}")
else:
    # 1. Read the metrics CSV and sort by DSC to find the worst results
    df = pd.read_csv(csv_path)
    df_sorted = df.sort_values(by='dsc', ascending=True)
    worst_cases = df_sorted.head(WORST_N_SAMPLES)

    # 2. Map the worst cases to their actual filenames in the prediction directory
    available_preds = os.listdir(pred_dir)
    target_files = []
    metrics_map = {}

    for _, row in worst_cases.iterrows():
        case_id = str(row['case'])

        # Find the matching file in the prediction directory
        # (This handles the fact that the CSV case name might not have the file extension)
        matching_files = [f for f in available_preds if case_id in f]

        if matching_files:
            target_file = matching_files[0]
            target_files.append(target_file)
            metrics_map[target_file] = {'dsc': row['dsc'], 'nsd': row['nsd']}
        else:
            print(f"Warning: Could not find prediction file for case {case_id}")

    # 3. Categorize the fetched worst files based on original logic
    files_2d = [f for f in target_files if '2D' in f or f.endswith(('.png', '.jpg', '.jpeg', '.npz')) and '3D' not in f]
    files_3d = [f for f in target_files if '3D' in f or f.endswith('.nii.gz')]

    # 4. Execute plotting with the injected metrics dictionary
    if files_2d:
        print("Rendering Worst 2D Modalities...")
        plot_comparisons(files_2d, "2D", metrics_dict=metrics_map)

    if files_3d:
        print("Rendering Worst 3D Modalities...")
        plot_comparisons(files_3d, "3D", metrics_dict=metrics_map)

In [ ]:
# Plot best score results
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import SimpleITK as sitk

# --- Configuration ---
base_dir = '/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data'
dataset_name = 'dental_caries_segmentation_processed'

# Directories
img_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/imgs'
gt_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/gts'
pred_dir = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/segs_og_ckpt'
csv_path = f'{base_dir}/{dataset_name}/MedSAM_test/RGB_Teeth/metrics_og_ckpt.csv'

# Define and create the missing output folder
test_output_folder = f'/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/experiments/{dataset_name}/og_ckpt'
out_path = f'{test_output_folder}/best_10_ds_teeth.png'


# Helper function to load arrays safely
def get_array(path):
    if not os.path.exists(path): return None
    try:
        if path.endswith('.npz'):
            data = np.load(path)
            return data[data.files[0]]
        elif path.endswith(('.png', '.jpg', '.jpeg')):
            if 'imgs' in path or 'images' in path:
                return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
            else:
                return cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        elif path.endswith(('.nii.gz', '.nii')):
            return sitk.GetArrayFromImage(sitk.ReadImage(path))
    except Exception as e:
        print(f"Error loading {path}: {e}")
    return None

def plot_comparisons(file_list, category_name, metrics_dict):
    if not file_list:
        print(f"No files found for {category_name}.")
        return

    # Create a figure with 4 columns per modality
    fig, axes = plt.subplots(len(file_list), 4, figsize=(22, 5 * len(file_list)))

    # Handle single row indexing edge-case
    if len(file_list) == 1: axes = [axes]

    # Add a custom master legend for the colors
    tp_patch = mpatches.Patch(color='yellow', label='Match (True Positive)')
    fn_patch = mpatches.Patch(color='lime', label='Missed (False Negative)')
    fp_patch = mpatches.Patch(color='red', label='Over-predicted (False Positive)')
    fig.legend(handles=[tp_patch, fn_patch, fp_patch], loc='upper center', ncol=3, fontsize=14, bbox_to_anchor=(0.5, 0.98))
    fig.suptitle(f"{dataset_name} - {category_name} Modalities - Best Results Analysis", fontsize=20, fontweight='bold', y=1.02)

    for i, filename in enumerate(file_list):
        img = get_array(os.path.join(img_dir, filename))
        gt = get_array(os.path.join(gt_dir, filename))
        pred = get_array(os.path.join(pred_dir, filename))

        if img is None or gt is None or pred is None:
            continue

        # Handle 3D slicing
        if img.ndim == 3 and img.shape[-1] != 3:
            mid = img.shape[0] // 2
            img, gt, pred = img[mid], gt[mid], pred[mid]

        # Standardize masks to binary (0 and 1)
        gt_bin = (gt > 0).astype(np.uint8)
        pred_bin = (pred > 0).astype(np.uint8)

        # Build the RGB Difference Map
        diff_map = np.zeros((*gt_bin.shape, 3), dtype=np.uint8)
        diff_map[(gt_bin == 1) & (pred_bin == 1)] = [255, 255, 0] # Yellow
        diff_map[(gt_bin == 1) & (pred_bin == 0)] = [0, 255, 0]   # Lime Green
        diff_map[(gt_bin == 0) & (pred_bin == 1)] = [255, 0, 0]   # Red

        # Clean title string
        clean_title = filename.replace('2DBox_', '').replace('3DBox_', '').replace('_demo.npz', '').replace('.nii.gz', '').replace('.npz', '')

        # Retrieve the scores for this specific image
        scores = metrics_dict.get(filename, {'dsc': 0.0, 'nsd': 0.0})
        score_text = f"DSC: {scores['dsc']:.4f} | NSD: {scores['nsd']:.4f}"

        row_axes = axes[i]

        # Col 1: Original
        row_axes[0].imshow(img, cmap='gray')
        row_axes[0].set_title(f"{clean_title}\nOriginal Image", fontsize=14)

        # Col 2: Ground Truth
        row_axes[1].imshow(img, cmap='gray')
        row_axes[1].imshow(np.ma.masked_where(gt_bin == 0, gt_bin), cmap='autumn', alpha=0.6)
        row_axes[1].set_title("Ground Truth", fontsize=14)

        # Col 3: Prediction (Now includes the scores)
        row_axes[2].imshow(img, cmap='gray')
        row_axes[2].imshow(np.ma.masked_where(pred_bin == 0, pred_bin), cmap='cool', alpha=0.6)
        row_axes[2].set_title(f"Model Prediction\n{score_text}", fontsize=14, color='darkred' if scores['dsc'] < 0.5 else 'black')

        # Col 4: Difference Map (Overlay)
        row_axes[3].imshow(img, cmap='gray')
        diff_mask = np.any(diff_map != [0, 0, 0], axis=-1)
        row_axes[3].imshow(np.ma.masked_where(~diff_mask, diff_map[..., 0]), cmap='gray')
        row_axes[3].imshow(diff_map, alpha=0.5)
        row_axes[3].set_title("Difference Map", fontsize=14)

        for ax in row_axes:
            ax.axis('off')

    plt.tight_layout()
    plt.subplots_adjust(top=0.92)

    # Save the file dynamically
    plt.savefig(out_path, bbox_inches='tight', dpi=300)
    print(f"Saved visualization to: {out_path}")

    plt.show()

# --- Main Execution Block ---
BEST_N_SAMPLES = 10 # Set how many of the best cases you want to see

if not os.path.exists(pred_dir):
    print(f"Prediction directory not found: {pred_dir}")
elif not os.path.exists(csv_path):
    print(f"Metrics CSV not found at: {csv_path}")
else:
    # 1. Read the metrics CSV and sort by DSC to find the best results
    df = pd.read_csv(csv_path)
    df_sorted = df.sort_values(by='dsc', ascending=False)
    best_cases = df_sorted.head(BEST_N_SAMPLES)

    # 2. Map the best cases to their actual filenames in the prediction directory
    available_preds = os.listdir(pred_dir)
    target_files = []
    metrics_map = {}

    for _, row in best_cases.iterrows():
        case_id = str(row['case'])

        # Find the matching file in the prediction directory
        # (This handles the fact that the CSV case name might not have the file extension)
        matching_files = [f for f in available_preds if case_id in f]

        if matching_files:
            target_file = matching_files[0]
            target_files.append(target_file)
            metrics_map[target_file] = {'dsc': row['dsc'], 'nsd': row['nsd']}
        else:
            print(f"Warning: Could not find prediction file for case {case_id}")

    # 3. Categorize the fetched best files based on original logic
    files_2d = [f for f in target_files if '2D' in f or f.endswith(('.png', '.jpg', '.jpeg', '.npz')) and '3D' not in f]
    files_3d = [f for f in target_files if '3D' in f or f.endswith('.nii.gz')]

    # 4. Execute plotting with the injected metrics dictionary
    if files_2d:
        print("Rendering Best 2D Modalities...")
        plot_comparisons(files_2d, "2D", metrics_dict=metrics_map)

    if files_3d:
        print("Rendering Best 3D Modalities...")
        plot_comparisons(files_3d, "3D", metrics_dict=metrics_map)

Correct metrics evaluation errors and rerun visualizations

In [ ]:
# Fix metrics calculation errors
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
!cp evaluation/compute_metrics.py evaluation/compute_metrics_binary.py
!sed -i "s/    segs = npz_seg\['segs'\]/    segs = (npz_seg['segs'] > 0).astype(np.uint8)/; s/    gts = npz_gt\['gts'\]/    gts = (npz_gt['gts'] > 0).astype(np.uint8)/" evaluation/compute_metrics_binary.py
!grep -n "> 0).astype" evaluation/compute_metrics_binary.py
T = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_test/RGB_Teeth"
!python evaluation/compute_metrics_binary.py -s {T}/segs_og_ckpt -g {T}/gts -csv_dir {T}/metrics_og_ckpt_binary.csv

In [ ]:
import pandas as pd, numpy as np
old = pd.read_csv(f"{T}/metrics_og_ckpt.csv"); new = pd.read_csv(f"{T}/metrics_og_ckpt_binary.csv")
m = old.merge(new, on="case", suffixes=("_old", "_new"))
m["n_boxes"] = [len(np.load(f"{T}/imgs/{c}")["boxes"]) for c in m.case]
print(m.groupby(m.n_boxes > 1)[["dsc_old", "dsc_new", "nsd_old", "nsd_new"]].mean())
print(m.sort_values("dsc_new").head(10)[["case", "n_boxes", "dsc_old", "dsc_new", "nsd_new"]])

In [ ]:
### metrics distribution of teeth ds on original lite med sam checkpoint

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the path to your metrics.csv file
csv_path = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_test/RGB_Teeth/metrics_og_ckpt_binary.csv"

# Load the dataset
df = pd.read_csv(csv_path)

# 1. Calculate and print the requested statistics
metrics = ['dsc', 'nsd']
print("--- Inference Statistics ---")
for metric in metrics:
    avg_val = df[metric].mean()
    med_val = df[metric].median()
    std_val = df[metric].std()

    print(f"\nMetric: {metric.upper()}")
    print(f"  Average (Mean): {avg_val:.4f}")
    print(f"  Median:         {med_val:.4f}")
    print(f"  Std Dev:        {std_val:.4f}")

# 2. Create the visualizations
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, metric in enumerate(metrics):
    # Boxplot shows the median, quartiles, and identifies outliers
    sns.boxplot(
        y=df[metric],
        ax=axes[i],
        color="lightgray",
        width=0.4,
        showfliers=False # Hide default outliers to prevent double-plotting with stripplot
    )

    # Stripplot overlays the actual distribution of all data points
    sns.stripplot(
        y=df[metric],
        ax=axes[i],
        color=sns.color_palette("husl", 2)[i],
        alpha=0.6,
        jitter=True,
        size=5
    )

    axes[i].set_title(f"Distribution of {metric.upper()}", fontsize=14, fontweight='bold')
    axes[i].set_ylabel("Score", fontsize=12)
    axes[i].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import Image, display
DATA_ROOT = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data"
DATASET = "dental_caries_segmentation_processed"
SUBSET = "RGB_Teeth"
PRED_FOLDER = "segs_og_ckpt"
OUT_DIR = f"/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/experiments/{DATASET}/og_ckpt"

In [ ]:
OUT_FILE = "Difference_Maps_10_ds_teeth_binary.png"
!python visualize_medsam.py \
    --data_root {DATA_ROOT} \
    --dataset {DATASET} \
    --subset {SUBSET} \
    --pred_folder {PRED_FOLDER} \
    --out_dir {OUT_DIR} \
    --out_file "Difference_Maps_10_ds_teeth_binary.png" \
    --mode random \
    --num_samples 10

# Update this variable to match the specific out_file you generated
image_path = f"{OUT_DIR}/{OUT_FILE}"

display(Image(filename=image_path))


In [ ]:
OUT_FILE = "worst_10_ds_teeth_binary.png"

!python visualize_medsam.py \
    --data_root {DATA_ROOT} \
    --dataset {DATASET} \
    --subset {SUBSET} \
    --pred_folder {PRED_FOLDER} \
    --out_dir {OUT_DIR} \
    --out_file "worst_10_ds_teeth_binary.png" \
    --mode worst \
    --num_samples 10 \
    --csv_name "metrics_og_ckpt_binary.csv"

# Update this variable to match the specific out_file you generated
image_path = f"{OUT_DIR}/{OUT_FILE}"

display(Image(filename=image_path))

In [ ]:
OUT_FILE = "best_7_ds_teeth_binary.png"
!python visualize_medsam.py \
    --data_root {DATA_ROOT} \
    --dataset {DATASET} \
    --subset {SUBSET} \
    --pred_folder {PRED_FOLDER} \
    --out_dir {OUT_DIR} \
    --out_file {OUT_FILE} \
    --mode best \
    --num_samples 7 \
    --csv_name "metrics_og_ckpt_binary.csv"


# Update this variable to match the specific out_file you generated
image_path = f"{OUT_DIR}/{OUT_FILE}"

display(Image(filename=image_path))

In [ ]:
from IPython.display import Image, display

# Update this variable to match the specific out_file you generated
image_path = f"{OUT_DIR}/Difference_Maps_10_ds_teeth_binary.png"

display(Image(filename=image_path))

Training

In [ ]:
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
T = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/"

!python pre_rgb_bbox_train.py \
  -img_path {T}/dental_caries_segmentation_processed/train/images \
  -gt_path {T}/dental_caries_segmentation_processed/train/masks \
  -bbox_format json \
  -bbox_json {T}/dental_caries_segmentation_processed/train/bboxes.json \
  -output_path {T}/dental_caries_segmentation_processed/MedSAM_train/RGB_Teeth/train_ready_npz \
  -modality RGB \
  -anatomy Teeth \
  -img_name_suffix .jpg \
  -gt_name_suffix .png \
  --binary_mask





In [ ]:
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
T = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_train/RGB_Teeth"

# Move comments outside the command block
# -npz_dir: path to the preprocessed npz training data
# -npy_dir: path to save the converted npy data for training
# -num_workers: number of workers for conversion in parallel

!python npz_to_npy.py \
  -npz_dir {T}/train_ready_npz \
  -npy_dir {T}/npy \
  -num_workers 4

In [ ]:
### Train Teeth ds with freezing the image encoder and prompt encoder, start from og checkpoint ###
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
T = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_train/RGB_Teeth"
!python train_one_gpu.py \
  -data_root {T}/npy \
  -pretrained_checkpoint "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/work_dir/LiteMedSAM/lite_medsam.pth" \
  -work_dir work_dir \
  -num_workers 4 \
  -batch_size 4 \
  -num_epochs 20 \
  --freeze_image_encoder \
  --freeze_prompt_encoder

In [ ]:
### Train Teeth ds with freezing the image encoder and prompt encoder, start from above checkpoint ###
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
T = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_train/RGB_Teeth"

!python train_one_gpu.py \
  -data_root {T}/npy \
  -pretrained_checkpoint "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/work_dir/LiteMedSAM/lite_medsam.pth" \
  -resume "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/work_dir/cont_og_frz_20e/medsam_lite_best.pth" \
  -work_dir "work_dir/cont_frz_20e_frz_20e-40e" \
  -num_workers 2 \
  -batch_size 4 \
  -num_epochs 40 \
  --freeze_image_encoder \
  --freeze_prompt_encoder

In [ ]:
### Train Teeth ds with freezing the image encoder and prompt encoder, start from og checkpoint, train Lora for mask decoder ###
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
T = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_train/RGB_Teeth"

!python train_one_gpu.py \
  -data_root {T}/npy \
  -pretrained_checkpoint '/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/work_dir/LiteMedSAM/lite_medsam.pth' \
  -work_dir work_dir/exp_frz_base_msk_decoder_lora_qv_r8 \
  -num_workers 2 \
  -batch_size 4 \
  -num_epochs 20 \
  --use_lora \
  --lora_scope mask_decoder \
  --lora_targets q,v \
  --lora_r 8 \
  --lora_alpha 16 \
  --lora_dropout 0.05

In [ ]:
### Train Teeth ds with freezing the image encoder and prompt encoder, start from og checkpoint, train Lora for mask decoder , hyperparams ###
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
T = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_train/RGB_Teeth"

!python train_one_gpu.py \
  -data_root {T}/npy \
  -pretrained_checkpoint '/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/work_dir/LiteMedSAM/lite_medsam.pth' \
  -work_dir work_dir/exp_frz_base_msk_decoder_lora_qv_r4 \
  -num_workers 2 \
  -batch_size 4 \
  -num_epochs 20 \
  --use_lora \
  --lora_scope mask_decoder \
  --lora_targets q,v \
  --lora_r 4 \
  --lora_alpha 16 \
  --lora_dropout 0.05

In [ ]:
### Continue Teeth ds LoRA run (mask decoder q,v r=4) from latest checkpoint
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
T = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_train/RGB_Teeth"

!python train_one_gpu.py \
  -data_root {T}/npy \
  -pretrained_checkpoint "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/work_dir/LiteMedSAM/lite_medsam.pth" \
  -resume "work_dir/exp_frz_base_msk_decoder_lora_qv_r4/medsam_lite_latest.pth" \
  --lora_checkpoint "work_dir/exp_frz_base_msk_decoder_lora_qv_r4/medsam_lite_lora_latest.pth" \
  -work_dir "work_dir/exp_frz_base_msk_decoder_lora_qv_r4_resume" \
  -num_workers 2 \
  -batch_size 4 \
  -num_epochs 20 \
  --use_lora \
  --lora_scope mask_decoder \
  --lora_targets q,v \
  --lora_r 4 \
  --lora_alpha 16 \
  --lora_dropout 0.05

In [ ]:
# Each run: converts the checkpoint, runs inference (skipped if all predictions exist),
# computes binary DSC/NSD, and writes summary stats to experiments/<dataset>/<run>/
# plus one row per run in experiments/<dataset>/runs_summary.csv.
import os, pandas as pd

REPO   = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh"
DEVICE = "cpu"          # "cuda" also works on a GPU runtime
FORCE_INFER = False     # True = re-run inference even if predictions already exist

# run name -> checkpoint (relative to MedSAM/). Uses the best (lowest training loss) checkpoint of each run.
RUNS = {
    "og_ckpt":    "work_dir/LiteMedSAM/lite_medsam.pth",                                    # original LiteMedSAM
    "frz_20e":    "work_dir/cont_og_frz_20e/medsam_lite_best.pth",                          # frozen enc+prompt, 20 ep
    "frz_40e":    "work_dir/cont_frz_20e_frz_20e-40e/medsam_lite_best.pth",                 # continued to 40 ep
    "lora_qv_r8": "work_dir/exp_frz_base_msk_decoder_lora_qv_r8/medsam_lite_best.pth",      # LoRA decoder q,v r=8
    "lora_qv_r4": "work_dir/exp_frz_base_msk_decoder_lora_qv_r4_resume/medsam_lite_best.pth",  # LoRA decoder q,v r=4
}
RUNS_TO_EVAL = list(RUNS)   # or a subset, e.g. ["lora_qv_r8", "lora_qv_r4"]

%cd {REPO}/MedSAM
for name in RUNS_TO_EVAL:
    ckpt = RUNS[name]
    if not os.path.isfile(ckpt):
        print(f"!! Skipping {name}: checkpoint not found at {ckpt}")
        continue
    force = "--force_infer" if FORCE_INFER else ""
    !python eval_run.py --run_name {name} --ckpt {ckpt} --repo {REPO} --device {DEVICE} {force}

runs = pd.read_csv(f"{REPO}/experiments/dental_caries_segmentation_processed/runs_summary.csv")
display(runs[["run", "n_cases", "dsc_mean", "dsc_median", "nsd_mean", "nsd_median",
              "dsc_single_box_mean", "dsc_multi_box_mean", "n_dsc_zero"]])


In [ ]:
from IPython.display import Image, display
REPO    = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh"
DATASET = "dental_caries_segmentation_processed"
SUBSET  = "RGB_Teeth"
T       = f"{REPO}/data/{DATASET}/MedSAM_test/{SUBSET}"
OUT     = f"{REPO}/experiments/{DATASET}/og_ckpt"
%cd {REPO}/MedSAM

!python plot_metrics_distribution.py \
    --csv {T}/metrics_og_ckpt.csv \
    --out_dir {OUT} \
    --out_file metrics_distribution_before_fix.png \
    --title "og_ckpt (original LiteMedSAM) - Teeth test - before metrics fix"
display(Image(filename=f"{OUT}/metrics_distribution_before_fix.png", width=1000))

In [ ]:
# Goes over runs listed in experiments/<dataset>/runs_summary.csv and each run's
# experiments/<dataset>/<run>/metrics_per_case.csv (binary metrics).
# Saves experiments/<dataset>/<run>/metrics_distribution_after_fix.png (+ _stats.csv) per run.
import os, pandas as pd
from IPython.display import Image, display
REPO    = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh"
DATASET = "dental_caries_segmentation_processed"
EXP     = f"{REPO}/experiments/{DATASET}"

RUN_TITLES = {   # plot titles; runs not listed here use their run name
    "og_ckpt":    "og_ckpt (original LiteMedSAM)",
    "frz_20e":    "frz_20e (frozen image+prompt encoders, 20 epochs)",
    "frz_40e":    "frz_40e (frozen image+prompt encoders, 40 epochs)",
    "lora_qv_r8": "lora_qv_r8 (LoRA mask decoder q,v, r=8)",
    "lora_qv_r4": "lora_qv_r4 (LoRA mask decoder q,v, r=4)",
}
RUNS_TO_PLOT = None   # None = every run in runs_summary.csv, or a list, e.g. ["og_ckpt", "lora_qv_r8"]

%cd {REPO}/MedSAM
runs = pd.read_csv(f"{EXP}/runs_summary.csv")["run"].tolist()
for name in (RUNS_TO_PLOT or runs):
    csv = f"{EXP}/{name}/metrics_per_case.csv"
    if not os.path.isfile(csv):
        print(f"!! Skipping {name}: {csv} not found (run Cell A for it first)")
        continue
    title = f"{RUN_TITLES.get(name, name)} - Teeth test - after metrics fix"
    !python plot_metrics_distribution.py --csv {csv} --out_dir {EXP}/{name} --out_file metrics_distribution_after_fix.png --title "{title}"
    display(Image(filename=f"{EXP}/{name}/metrics_distribution_after_fix.png", width=1000))

In [ ]:
!python diagnose_bbox_survival.py \
  -npz_dir /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_train/RGB_Teeth/train_ready_npz \
  -out_csv /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/work_dir/bbox_survival_report.csv \
  --save_failed_vis \
  -vis_dir /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/work_dir/bbox_survival_failed_vis

In [ ]:
!python diagnose_bbox_survival.py \
  -data_root "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_train/RGB_Teeth/npy" \
  -out_csv "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/work_dir/bbox_survival_train_one_gpu_flow.csv" \
  --save_failed_vis \
  -vis_dir "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/work_dir/bbox_survival_failed_vis_train_one_gpu_flow"

In [ ]:
!grep -n "fg_orig = 0" /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/diagnose_bbox_survival.py

In [ ]:
!grep -n "def process_npz_file\|def process_npy_pair\|fg_orig = 0" /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/diagnose_bbox_survival.py

Debugging Problems with checking how many lesions disappear at 256 px



In [ ]:
"Sync error - do not run again"
SRC = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_train/RGB_Teeth/train_ready_npz"
!du -sh "{SRC}"
!mkdir -p /content/train_ready_npz
!rsync -a --info=progress2 "{SRC}/" /content/train_ready_npz/
!ls /content/train_ready_npz | wc -l

In [ ]:
"Run this cell if file copying fails"
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
"Fails at 1034 file - abandon file for now"
SRC = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_train/RGB_Teeth/train_ready_npz"
!rsync -a --info=progress2 --exclude='RGB_Teeth_1034_*' "{SRC}/" /content/train_ready_npz/
!ls /content/train_ready_npz | wc -l

In [ ]:
!ls -la "{SRC}" | grep -E "RGB_Teeth_10(2|3|4)"

In [ ]:
%cd /content

In [ ]:
# Copy files excluding suspect file (that crashed multiple times)
SRC = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_train/RGB_Teeth/train_ready_npz"
!rsync -a --info=progress2 --exclude='RGB_Teeth_1034_*' "{SRC}/" /content/train_ready_npz/
!ls /content/train_ready_npz | wc -l

In [ ]:
# Copy the script to Colab disk
!cp /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/diagnose_bbox_survival.py /content/
!grep -c "lesion_vanished_neighbor_in_box" /content/diagnose_bbox_survival.py

In [ ]:
# Check the copies
import numpy as np, glob
bad = []
for f in sorted(glob.glob("/content/train_ready_npz/*.npz")):
    try:
        np.load(f).files
    except Exception as e:
        bad.append((f, e))
print("bad files:", len(bad)); print(bad[:5])

In [ ]:
# Run the diagnostic from Colab's disk
!python /content/diagnose_bbox_survival.py \
  -npz_dir /content/train_ready_npz \
  -out_csv /content/bbox_survival_per_lesion.csv \
  --save_failed_vis \
  -vis_dir /content/bbox_survival_failed_vis

In [ ]:
# Copy the results back to Drive
OUT = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/work_dir/bbox_survival_per_lesion"
!mkdir -p "{OUT}"
!cp /content/bbox_survival_per_lesion.csv "{OUT}/"
!cp -r /content/bbox_survival_failed_vis "{OUT}/"
!ls "{OUT}/bbox_survival_failed_vis" | wc -l

In [ ]:
# Check suspect file on its own
!cp "{SRC}/RGB_Teeth_1034_jpg.rf.26000309095e7f3e18287597e93ea8c7.npz" /content/train_ready_npz/

In [ ]:
# Check the file isn't corrupt
import numpy as np
f = "/content/train_ready_npz/RGB_Teeth_1034_jpg.rf.26000309095e7f3e18287597e93ea8c7.npz"
d = np.load(f); print(d.files, d["gts"].shape, len(d["boxes"]))

# Decide is the data needs pre-processing re-do
We do this by counting thin lesions that have no box

In [ ]:
# move out of Drive and copy only the masks and box files
%cd /content
D = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed"
for split in ["train", "test"]:
    !mkdir -p /content/step1/{split}
    !rsync -a --info=progress2 "{D}/{split}/masks" "{D}/{split}/bboxes.json" /content/step1/{split}/
!ls /content/step1/train/masks | wc -l; ls /content/step1/test/masks | wc -l
!cp /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/count_unboxed_lesions.py /content/

In [ ]:
# Run on both test and train
!python /content/count_unboxed_lesions.py -split_dir /content/step1/train -out_csv /content/step1/unboxed_train.csv
!python /content/count_unboxed_lesions.py -split_dir /content/step1/test  -out_csv /content/step1/unboxed_test.csv

In [ ]:
# Copy the results to Drive
OUT = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/work_dir/unboxed_lesions"
!mkdir -p "{OUT}"
!cp /content/step1/unboxed_*.csv "{OUT}/"

# Creating a Validation dataset
Splitting the test dataset into test and validation

In [ ]:
# Copying the datasets into Collab local disk
%cd /content
D = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/MedSAM_test/RGB_Teeth"
L = "/content/work/data/dental_caries_segmentation_processed/MedSAM_test/RGB_Teeth"
!mkdir -p "{L}"
!rsync -a --info=progress2 --include='imgs/***' --include='gts/***' --include='segs_*/***' --exclude='*' "{D}/" "{L}/"
!ls "{L}"; ls "{L}/imgs" | wc -l; ls "{L}/gts" | wc -l
!cp /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/split_val_test.py /content/

In [ ]:
# Splitting the test dataset
!python /content/split_val_test.py -test_dir "{L}" -out_root /content/work/data

In [ ]:
# Copying the results to Drive
DD = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data"
!rsync -a --info=progress2 /content/work/data/dental_caries_val42 /content/work/data/dental_caries_test63 /content/work/data/split_val42_test63.csv "{DD}/"
!ls "{DD}/dental_caries_val42/MedSAM_test/RGB_Teeth/imgs" | wc -l; ls "{DD}/dental_caries_test63/MedSAM_test/RGB_Teeth/imgs" | wc -l

# Re-score B0 and the old runs on each part
For comparison after the dataset split

In [ ]:
# Copying the code locally
%cd /content
REPO_D = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh"
!mkdir -p /content/code/MedSAM
!rsync -a --exclude='work_dir' --exclude='__pycache__' "{REPO_D}/MedSAM/" /content/code/MedSAM/
!ls /content/code/MedSAM | head
%cd /content/code/MedSAM

In [ ]:
# Score all five runs on split dataset
CK = f"{REPO_D}/MedSAM/work_dir"
RUNS = {
    "og_ckpt":    f"{CK}/LiteMedSAM/lite_medsam.pth",
    "frz_20e":    f"{CK}/cont_og_frz_20e/medsam_lite_best.pth",
    "frz_40e":    f"{CK}/cont_frz_20e_frz_20e-40e/medsam_lite_best.pth",
    "lora_qv_r8": f"{CK}/exp_frz_base_msk_decoder_lora_qv_r8/medsam_lite_best.pth",
    "lora_qv_r4": f"{CK}/exp_frz_base_msk_decoder_lora_qv_r4_resume/medsam_lite_best.pth",
}
for ds in ["dental_caries_val42", "dental_caries_test63"]:
    for name, ck in RUNS.items():
        !python eval_run.py --run_name {name} --ckpt "{ck}" --repo /content/work --dataset {ds}

In [ ]:
# Print scores and save back to Drive
import pandas as pd
cols = ["run","n_cases","dsc_mean","dsc_median","nsd_mean","nsd_median","dsc_single_box_mean","dsc_multi_box_mean","n_dsc_zero"]
for ds in ["dental_caries_val42", "dental_caries_test63"]:
    print(ds); display(pd.read_csv(f"/content/work/experiments/{ds}/runs_summary.csv")[cols])
!rsync -a /content/work/experiments/ "{REPO_D}/experiments/"

# Zoomed on lesions check
Check zero-shot on zoomed in lesion in large images. This is a desicion point for us on wether or not cropped and zoomed in lesions should be added into our pipline to improve our results.

In [ ]:
# Pulling what's needed in Collab local disk
%cd /content
REPO_D = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh"
!mkdir -p /content/code/MedSAM /content/work/data /content/work/experiments
!rsync -a --exclude='work_dir' --exclude='__pycache__' "{REPO_D}/MedSAM/" /content/code/MedSAM/
!rsync -a --info=progress2 "{REPO_D}/data/dental_caries_val42" /content/work/data/
!rsync -a "{REPO_D}/experiments/dental_caries_val42" /content/work/experiments/
V = "/content/work/data/dental_caries_val42/MedSAM_test/RGB_Teeth"
!ls "{V}/imgs" | wc -l; ls /content/work/experiments/dental_caries_val42
%cd /content/code/MedSAM

In [ ]:
# Zoom inference on validation photos
OG = f"{REPO_D}/MedSAM/work_dir/LiteMedSAM/lite_medsam.pth"
!python infer_zoom.py -i "{V}/imgs" -o "{V}/segs_zoom_z1" -lite_medsam_checkpoint_path "{OG}" \
    -reference_dir "{V}/segs_og_ckpt" -device cpu

In [ ]:
# Score inference on validation dataset
!python eval_run.py --run_name zoom_z1 --ckpt "{OG}" --repo /content/work --dataset dental_caries_val42

In [ ]:
# Apply the decision rule, for DSC and then NSD
E = "/content/work/experiments/dental_caries_val42"
SPLIT = f"{REPO_D}/data/split_val42_test63.csv"
!ls -la "{SPLIT}"
for m in ["dsc", "nsd"]:
    !python compare_paired.py -candidate {E}/zoom_z1/metrics_per_case.csv -reference {E}/og_ckpt/metrics_per_case.csv \
        -split_csv "{SPLIT}" -metric {m} -out_csv {E}/zoom_z1/paired_{m}.csv

In [ ]:
# Save results into Drive
!rsync -a /content/work/experiments/ "{REPO_D}/experiments/"
!rsync -a "{V}/segs_zoom_z1" "{REPO_D}/data/dental_caries_val42/MedSAM_test/RGB_Teeth/"

In [ ]:
# Visual check of zoom results
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
REPO_D = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh"
V = f"{REPO_D}/data/dental_caries_val42/MedSAM_test/RGB_Teeth"
E = f"{REPO_D}/experiments/dental_caries_val42"
for p in [f"{E}/zoom_z1/paired_dsc.csv", f"{V}/segs_zoom_z1", f"{V}/segs_og_ckpt"]:
    print(("OK      " if os.path.exists(p) else "MISSING ") + p)

d = pd.read_csv(f"{E}/zoom_z1/paired_dsc.csv").sort_values("diff")
pick = list(d.case[:1]) + list(d.case[-3:])            # the one loss + the three biggest wins
diff = d.set_index("case")["diff"]
fig, ax = plt.subplots(len(pick), 3, figsize=(15, 5 * len(pick)))
for r, c in enumerate(pick):
    with np.load(f"{V}/imgs/{c}") as f:
        img, b = f["imgs"], f["boxes"]
    H, W = img.shape[:2]
    x0, y0 = max(int(b[:, 0].min()) - 100, 0), max(int(b[:, 1].min()) - 100, 0)
    x1, y1 = min(int(b[:, 2].max()) + 100, W), min(int(b[:, 3].max()) + 100, H)
    masks = [("ground truth", np.load(f"{V}/gts/{c}")["gts"]),
             ("B0", np.load(f"{V}/segs_og_ckpt/{c}")["segs"]),
             ("zoom", np.load(f"{V}/segs_zoom_z1/{c}")["segs"])]
    for k, (title, m) in enumerate(masks):
        ax[r, k].imshow(img[y0:y1, x0:x1])
        ax[r, k].imshow(np.ma.masked_equal(m[y0:y1, x0:x1] > 0, False), alpha=0.45, cmap="autumn")
        ax[r, k].set_title(f"{title} | {c[:18]} | {W}x{H} | diff {diff[c]:+.2f}", fontsize=10)
        ax[r, k].axis("off")
plt.tight_layout()
plt.savefig(f"{E}/zoom_z1/visual_check.png", dpi=80)
plt.show()

In [ ]:
# Visual check of zoom results
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
REPO_D = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh"
V = f"{REPO_D}/data/dental_caries_val42/MedSAM_test/RGB_Teeth"
E = f"{REPO_D}/experiments/dental_caries_val42"
split = pd.read_csv(f"{REPO_D}/data/split_val42_test63.csv").set_index("file")

d = pd.read_csv(f"{E}/zoom_z1/paired_dsc.csv").sort_values("diff")
diff = d.set_index("case")["diff"]
rows = [(d.case.iloc[0], "Zoom's only loss")] + \
       [(c, f"Biggest win #{i}") for i, c in enumerate(d.case.iloc[::-1][:3], start=1)]

fig, ax = plt.subplots(len(rows), 3, figsize=(15, 5 * len(rows)))
fig.suptitle("Zero-shot LiteMedSAM: whole photo (B0) vs one crop per box (zoom), validation photos\n"
             "Rows: the one photo where zoom scored lower, then the three photos where zoom gained most (DSC)",
             fontsize=14, y=1.0)
for r, (c, label) in enumerate(rows):
    with np.load(f"{V}/imgs/{c}") as f:
        img, b = f["imgs"], f["boxes"]
    H, W = img.shape[:2]
    size = "large" if split.loc[c, "size_group"] == "large" else "small"
    x0, y0 = max(int(b[:, 0].min()) - 100, 0), max(int(b[:, 1].min()) - 100, 0)
    x1, y1 = min(int(b[:, 2].max()) + 100, W), min(int(b[:, 3].max()) + 100, H)
    masks = [("Ground truth", np.load(f"{V}/gts/{c}")["gts"]),
             ("B0 (whole photo)", np.load(f"{V}/segs_og_ckpt/{c}")["segs"]),
             ("Zoom (crop per box)", np.load(f"{V}/segs_zoom_z1/{c}")["segs"])]
    for k, (title, m) in enumerate(masks):
        ax[r, k].imshow(img[y0:y1, x0:x1])
        ax[r, k].imshow(np.ma.masked_equal(m[y0:y1, x0:x1] > 0, False), alpha=0.45, cmap="autumn")
        ax[r, k].set_title(title, fontsize=11)
        ax[r, k].axis("off")
    ax[r, 0].text(-0.04, 0.5, f"{label}\n{c[:18]}\n{W}×{H} px ({size} photo)\nDSC change {diff[c]:+.2f}",
                  transform=ax[r, 0].transAxes, ha="right", va="center", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{E}/zoom_z1/visual_check.png", dpi=80, bbox_inches="tight")
plt.show()

In [ ]:
import json
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# --- paths ---
mask_path = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/test/masks/3125_jpg.rf.cafbec6476ccb0204f295c9fc3163388.png"
bbox_json_path = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/data/dental_caries_segmentation_processed/test/bboxes.json"

# --- load JSON ---
with open(bbox_json_path, "r") as f:
    bbox_map = json.load(f)

# JSON keys usually look like filename stem without extension.
# Example: "3434_jpg.rf.c030dd17d61cd263f37365e2b77b6e08"
mask_stem = Path(mask_path).stem

item = bbox_map.get(mask_stem)
if item is None:
    # fallback: try with file name including extension
    item = bbox_map.get(Path(mask_path).name)
if item is None:
    raise ValueError(f"No bbox entry found for {mask_path}")

# --- load mask ---
mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
if mask is None:
    raise ValueError(f"Could not read mask: {mask_path}")

# make binary mask visible
mask_vis = np.where(mask > 0, 255, 0).astype(np.uint8)
vis = cv2.cvtColor(mask_vis, cv2.COLOR_GRAY2RGB)

# --- draw boxes ---
boxes = item.get("bbox", [])
for b in boxes:
    label = b.get("label", "obj")
    x1 = int(b["xmin"])
    y1 = int(b["ymin"])
    x2 = int(b["xmax"])
    y2 = int(b["ymax"])

    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
    #cv2.putText(vis, label, (x1, max(0, y1 - 8)),
     #           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

# --- show ---
plt.figure(figsize=(10, 8))
plt.imshow(vis[:, :, ::-1])  # BGR -> RGB for matplotlib
plt.title(f"Mask + boxes: {mask_stem}")
plt.axis("off")
plt.show()

# Check box jitter
To check if zoom still beats baseline

In [ ]:
# Pull relevant files into Collab local disk
%cd /content
REPO_D = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh"
!mkdir -p /content/code/MedSAM /content/work/data /content/work/experiments
!rsync -a --exclude='work_dir' --exclude='__pycache__' "{REPO_D}/MedSAM/" /content/code/MedSAM/
!rsync -a --info=progress2 "{REPO_D}/data/dental_caries_val42" /content/work/data/
!rsync -a "{REPO_D}/experiments/dental_caries_val42" /content/work/experiments/
%cd /content/code/MedSAM
!grep -c "box_perturb" infer_zoom.py
V  = "/content/work/data/dental_caries_val42/MedSAM_test/RGB_Teeth"
OG = f"{REPO_D}/MedSAM/work_dir/LiteMedSAM/lite_medsam.pth"

In [ ]:
# Zoom only on photos above 2,000 px
!python infer_zoom.py -i "{V}/imgs" -o "{V}/segs_Z" -lite_medsam_checkpoint_path "{OG}" \
    -zoom_above 2000 -reference_dir "{V}/segs_og_ckpt" -device cpu
!python eval_run.py --run_name Z --ckpt "{OG}" --repo /content/work --dataset dental_caries_val42

In [ ]:
# Box jitter (baseline vs zoom)
for p, tag in [(0.1, "p10"), (0.2, "p20")]:
    for name, za in [("B0", 100000), ("Z", 2000)]:
        run = f"{name}_{tag}"
        !python infer_zoom.py -i "{V}/imgs" -o "{V}/segs_{run}" -lite_medsam_checkpoint_path "{OG}" \
            -zoom_above {za} -box_perturb {p} -perturb_seed 0 -device cpu
        !python eval_run.py --run_name {run} --ckpt "{OG}" --repo /content/work --dataset dental_caries_val42

In [ ]:
# Check box jitter influence
import pandas as pd
from compare_paired import summarize
E = "/content/work/experiments/dental_caries_val42"
split = pd.read_csv(f"{REPO_D}/data/split_val42_test63.csv").rename(columns={"file": "case"})[["case", "size_group"]]
def load(run): return pd.read_csv(f"{E}/{run}/metrics_per_case.csv")[["case", "dsc"]].merge(split, on="case")
def paired(cand, ref, group=None):
    m = load(cand).merge(load(ref)[["case", "dsc"]], on="case", suffixes=("_c", "_r"))
    if group: m = m[m.size_group == group]
    return summarize((m.dsc_c - m.dsc_r).to_numpy(), 0.03, 0.05)

print("Mean DSC on validation:")
for r in ["og_ckpt", "Z", "B0_p10", "Z_p10", "B0_p20", "Z_p20"]:
    print(f"  {r:<8} {load(r).dsc.mean():.3f}")
k = paired("Z_p20", "B0_p20", "large")
print(f"\nRule 1  Z vs B0, large photos, 20% boxes: {k['wins']}/{k['losses']} wins/losses, mean {k['mean_diff']:+.3f}, "
      f"sign p {k['sign_p_one_sided']:.4f}  ->  {'KEEP training on crops' if k['passes'] else 'STOP: rethink the crop rule'}")
for cand, ref in [("B0_p20", "og_ckpt"), ("Z_p20", "Z")]:
    d = paired(cand, ref)["mean_diff"]
    print(f"Rule 2  {ref:>7} -> {cand}: mean DSC change {d:+.3f}  ->  {'RAISE -bbox_shift' if d < -0.05 else 'keep -bbox_shift 5'}")

In [ ]:
# Save results to Drive
!rsync -a /content/work/experiments/ "{REPO_D}/experiments/"
for r in ["Z", "B0_p10", "Z_p10", "B0_p20", "Z_p20"]:
    !rsync -a "{V}/segs_{r}" "{REPO_D}/data/dental_caries_val42/MedSAM_test/RGB_Teeth/"